# 1D CNN for Viral Recombination Breakpoint Detection

End-to-end pipeline that learns to localise recombination breakpoints from
aligned viral sequences. Unlike feature-based approaches (RDPML.ipynb,
RDPML_PSNN.ipynb), this model works directly on raw nucleotide data using
one-hot encoding plus explicit parent-comparison signals.

## Pipeline overview

| Step | What happens | Where |
|------|--------------|-------|
| Parse | Read FASTA + SANTA CSVs to get `(recomb, parent1, parent2)` triplets and their breakpoint positions | `parse_simulation` |
| Encode | One-hot the triplet (15 channels) and append 3 comparison channels surfacing which parent the recombinant matches | `encode_triplet` |
| Label | Mark a small +/- window around each true breakpoint as positive | `generate_labels` |
| Split | Group-split by FASTA file so events from the same simulation never straddle train and val | Cell below |
| Train | Multi-scale 1D CNN with focal loss for the heavy class imbalance | `build_cnn` |
| Evaluate | Peak detection on the per-position output, matched to true breakpoints with +/- tolerance | `evaluate_peaks` |

## Models

1. **Multi-scale 1D CNN** (primary). Parallel conv branches with kernels
   `{3, 7, 15, 31}` capture short- and medium-range patterns; a small
   integration stack and 1x1 head produce a per-position probability.
2. **Convolutional autoencoder** (secondary, exploratory). Trained on
   non-recombinant sequences; reconstruction error is intended as an
   anomaly signal at breakpoint regions. **Note**: the AE has known design
   issues (see TODO.md) and is not the focus of current work.

## Why explicit comparison channels?

The recombination signal at a breakpoint is a flip in *which parent the
recombinant matches*. Forcing the model to rediscover that comparison
from raw one-hots is wasteful. Three extra channels (`match_p1`,
`match_p2`, `informative`) expose the signal directly and tend to make
the training objective much easier.

## Why peak-based evaluation?

A breakpoint is a single position, not a region. Counting every
above-threshold position as a separate prediction inflates both true
and false positive counts when the model produces broad ridges.
`scipy.signal.find_peaks` with a minimum-separation constraint collapses
each ridge to a single peak, which is what we actually want to count
against the two ground-truth breakpoint positions.

In [1]:
# Core libraries
import numpy as np
import pandas as pd
import tensorflow as tf

# Enable GPU memory growth so TF only allocates VRAM as needed (default
# behaviour pre-allocates ~all available, which is wasteful and hides
# how much the model actually uses). Must run before any GPU op.
for _gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(_gpu, True)
    except RuntimeError:
        # set_memory_growth must precede GPU init; ignore if already initialised
        pass

from pathlib import Path
import os
import warnings
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# TensorFlow/Keras
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, UpSampling1D,
    Dropout, BatchNormalization, Dense, Flatten,
    Concatenate, GlobalAveragePooling1D, Reshape,
    Add, Activation, Lambda,
)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam, AdamW
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras import backend as K

# Scikit-learn
from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score,
    roc_curve, auc, precision_recall_curve, average_precision_score
)

# SciPy (peak detection for breakpoint evaluation)
from scipy.signal import find_peaks

# BioPython for FASTA parsing
from Bio import SeqIO

# Progress tracking (tqdm.auto falls back to plain text bar if widgets unavailable)
from tqdm.auto import tqdm

# Reproducibility
np.random.seed(42)
tf.random.set_seed(42)

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")


2026-05-05 17:16:46.220985: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777994206.328665   18926 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777994206.359936   18926 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-05 17:16:46.515056: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow version: 2.18.1
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# GPU sanity check.
# Confirms TensorFlow can actually run ops on the GPU, not just *see* one.
# Apple Silicon (Metal): pip install tensorflow-metal.
# Linux/Windows (CUDA):  pip install tensorflow[and-cuda]==2.18.* (ships its own CUDA/cuDNN).

import platform

print(f"Platform: {platform.platform()}  ({platform.machine()})")
print(f"TensorFlow: {tf.__version__}")

gpus = tf.config.list_physical_devices('GPU')
print(f"GPUs visible to TF: {gpus}")

if not gpus:
    print("\n[!] No GPU detected.")
    print("    Apple Silicon: pip install tensorflow-metal")
    print("    Linux/CUDA:    pip install tensorflow[and-cuda]==2.18.*")
    print("    and restart the kernel.")
else:
    # Force a real op onto the GPU and check where it ran. TF silently
    # falls back to CPU for unsupported ops, so a non-empty GPU list is
    # not a sufficient signal on its own.
    with tf.device('/GPU:0'):
        a = tf.random.normal([4096, 4096])
        b = tf.matmul(a, a)
    print(f"matmul executed on: {b.device}")
    if 'GPU' in b.device:
        print("[OK] GPU acceleration is active.")
    else:
        print("[!] Op fell back to CPU despite GPU being visible.")


Platform: Linux-6.6.87.2-microsoft-standard-WSL2-x86_64-with-glibc2.39  (x86_64)
TensorFlow: 2.18.1
GPUs visible to TF: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


matmul executed on: /job:localhost/replica:0/task:0/device:GPU:0
[OK] GPU acceleration is active.


I0000 00:00:1777994211.062425   18926 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5562 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3070, pci bus id: 0000:26:00.0, compute capability: 8.6


In [3]:
# Configuration

# Data paths
DATA_ROOT = Path("dataRaw")
TRAIN_DIRS = ["XML-1", "XML-2", "XML-3", "XML-4", "XML-5"]
TEST_DIR = "UnseenTestSet"

# Sequence encoding
# Bumped 4000 → 10000 in run #7 to cover full HIV-class genomes (~9.7 kb).
# Bumped 10000 → 32000 in run #24 — diagnostic at end of 2026-05-05 session
# revealed UnseenTestSet sequences are ~30 kb (val 10.8 kb mean) so 74.4%
# of test bp_end values were past the truncation horizon. 32000 captures
# all observed test breakpoints (max bp_end ≈ 30,143). Cache invalidates
# automatically because cache key includes MAX_SEQ_LEN.
MAX_SEQ_LEN = 32000
NUCLEOTIDES = ['A', 'T', 'G', 'C', '-']
N_CHANNELS = len(NUCLEOTIDES)  # 5 (one-hot dimension per sequence)
# Run #17 (REVERTED): tested 6 windows (50,100,200,500,1000,2000); val
# improved slightly but test F1 dropped 0.172 → 0.161 — see Experiment Log.
N_MAXCHI_WINDOWS = 4
N_INPUT_CHANNELS = 3 * N_CHANNELS + 3 + N_MAXCHI_WINDOWS  # 22
MAXCHI_WINDOWS = (50, 100, 200, 500)

# Label generation (LEGACY: per-position Gaussian/breakpoint/region targets).
# Run #19+ uses top-K coordinate regression instead — targets come from
# meta['bp_start','bp_end'] directly via make_topk_targets, NOT from y.
# These constants are still used by load_dataset to populate the cache so
# legacy modes are recoverable, but they don't drive training under top-K.
LABEL_MODE = 'gaussian'     # 'gaussian' | 'breakpoint' | 'region'
LABEL_SIGMA = 20            # Gaussian half-width (bp) for soft targets (Run #18 tested 10 paired w/ POS_WEIGHT=140 — REVERTED, see Experiment Log)
BP_WINDOW = 10              # Used only when LABEL_MODE == 'breakpoint'
TOLERANCE = 200             # +/- bp tolerance for evaluation

# Top-K coordinate regression head (run #19+)
K_TOPK = 2              # Two breakpoints per recombination event
# Target smoothing for top-K. Run #19 used 0 (hard one-hot, INCONCLUSIVE — model
# stalled at val_match_rate=0.33, best epoch=1). Run #20+ uses σ=5: targets are
# normalized Gaussian peaks spreading ~10 positions of mass per head; smoother
# gradient than one-hot, still concentrated enough to encourage sharp argmax.
TOPK_TARGET_SIGMA = 5
# Run #21+ — edge-buffer suppression. Runs #19/#20 collapsed to predicting
# the literal sequence boundaries (head-0 → position 0, head-1 → 9999) due
# to BN+padding artifact + target-clamping pile-up. Banning the first/last
# EDGE_BUFFER positions in the softmax logits forces predictions into the
# interior, breaking the boundary attractor. Targets in [0, EDGE_BUFFER) ∪
# [MAX_SEQ_LEN - EDGE_BUFFER, MAX_SEQ_LEN) are also skipped (set to all-zero)
# so cross-entropy doesn't penalise the model for not predicting in suppressed
# regions.
EDGE_BUFFER = 50

# Training
BATCH_SIZE = 2  # Run #24: dropped 8→2 alongside MAX_SEQ_LEN 10k→32k.
                # 3.2× longer sequences in 8GB VRAM forced the drop;
                # if no OOM, future runs can probe BATCH_SIZE=4.
EPOCHS = 100
LR = 1e-4
VAL_SPLIT = 0.15

# Focal loss (LEGACY — kept for reference; weighted_bce was the per-position active loss)
FOCAL_ALPHA = 0.25
FOCAL_GAMMA = 2.0

# Class imbalance: POS_WEIGHT is hardcoded in cell-12 regardless of the
# data-implied value. Used by the LEGACY per-position loss only; top-K
# uses categorical cross-entropy with no class-weight scalar.
# Run #8 set 70 (vs data-implied 82.62 at MAX_SEQ_LEN=10k).
# Run #24 rescales to 178 (vs data-implied ~210 at MAX_SEQ_LEN=32k),
# preserving the same 0.847× factor.
POS_WEIGHT = 178.0

print(f"Max sequence length: {MAX_SEQ_LEN}")
print(f"Input channels (triplet + comparisons + MaxChi): {N_INPUT_CHANNELS}")
print(f"MaxChi windows (bp): {MAXCHI_WINDOWS}")
print(f"Label mode (legacy): {LABEL_MODE} (sigma={LABEL_SIGMA})")
print(f"K (top-K head):      {K_TOPK}")
print(f"Training directories: {TRAIN_DIRS}")
print(f"pos_weight (legacy, per-position only): {POS_WEIGHT}")


Max sequence length: 32000
Input channels (triplet + comparisons + MaxChi): 22
MaxChi windows (bp): (50, 100, 200, 500)
Label mode (legacy): gaussian (sigma=20)
K (top-K head):      2
Training directories: ['XML-1', 'XML-2', 'XML-3', 'XML-4', 'XML-5']
pos_weight (legacy, per-position only): 178.0


## Data parsing and preprocessing

Each SANTA simulation produces three correlated files:

| File | Contents |
|------|----------|
| `*.fa` | FASTA alignment, ~100 sequences per file |
| `*.faSimVSRealCompare.csv` | Ground truth: `ActualRecomb` (sequence ID), `SimBPStart`, `SimBPEnd` |
| `*.faRecombIdentifyStats.csv` | Three rows per recombination event with `ISeqs(A)` listing triplet members |

For each event we extract the **triplet** -- recombinant + 2 parents.
The recombinant is the row whose `ISeqs(A)` includes `ActualRecomb`;
the other two rows give the parent IDs.

### One-hot encoding (with comparison channels)

Each position is encoded as 18 channels:

- **0-4**: recombinant nucleotide one-hot over `[A, T, G, C, -]`
- **5-9**: parent 1 one-hot
- **10-14**: parent 2 one-hot
- **15**: `match_p1` -- recombinant matches parent 1 at this position
- **16**: `match_p2` -- recombinant matches parent 2
- **17**: `informative` -- the two parents differ here

Padding positions (past the end of any sequence) have all comparison
channels set to zero via a validity mask, so the model gets no spurious
signal where there is no data.

In [4]:
def one_hot_encode(sequence, max_length=MAX_SEQ_LEN):
    """One-hot encode a nucleotide sequence (A/T/G/C/gap) with padding.

    Args:
        sequence: Nucleotide string.
        max_length: Pad or truncate to this length.

    Returns:
        np.ndarray of shape (max_length, 5).
    """
    nuc_idx = {'A': 0, 'T': 1, 'G': 2, 'C': 3, '-': 4}
    encoded = np.zeros((max_length, N_CHANNELS), dtype=np.float32)
    for i, nuc in enumerate(sequence[:max_length].upper()):
        idx = nuc_idx.get(nuc, 4)  # Unknown nucleotides treated as gaps
        encoded[i, idx] = 1.0
    return encoded


def _seq_to_index(sequence, max_length=MAX_SEQ_LEN):
    """Per-position nucleotide index, with -1 marking padding (post sequence end)."""
    nuc_idx = {'A': 0, 'T': 1, 'G': 2, 'C': 3, '-': 4}
    idx = np.full(max_length, -1, dtype=np.int16)
    for i, nuc in enumerate(sequence[:max_length].upper()):
        idx[i] = nuc_idx.get(nuc, 4)
    return idx


def _maxchi_features(parental_signal, windows=MAXCHI_WINDOWS, length=MAX_SEQ_LEN):
    """Per-position right-minus-left running-mean disparity of the parental
    signal at multiple window sizes. Equivalent to the MaxChi/CUSUM
    statistic for parental-switch detection.

    Args:
        parental_signal: 1-D float array of length `length`. +1 where
            recomb matches P1 only, -1 where it matches P2 only, 0
            elsewhere (both match, neither match, or padding).
        windows: tuple of window sizes (in bp).

    Returns:
        np.ndarray of shape (length, len(windows)). Channel k holds
        right_mean(parental_signal[p:p+w_k]) - left_mean(parental_signal[p-w_k:p]).
    """
    out = np.zeros((length, len(windows)), dtype=np.float32)
    for k, w in enumerate(windows):
        # Pad with zeros so left/right means at boundaries treat
        # off-the-end data as zero contribution.
        padded = np.pad(parental_signal, (w, w), mode='constant')
        csum = np.cumsum(padded, dtype=np.float64)
        # Indices in the padded array map to original p via +w shift.
        # left_mean at p = mean(parental_signal[p-w:p]) =
        #     (csum[p-1+w] - csum[p-1+w - w]) / w   -- using cumsum[i] = sum(padded[:i+1])
        # Use the convention csum[-1] := 0; numpy cumsum without initial zero requires care.
        # Easier: reconstruct via prefix sums with a leading zero.
        prefix = np.concatenate(([0.0], csum))  # prefix[i] = sum(padded[:i])
        # left_mean[p] = (prefix[p+w] - prefix[p]) / w  for p in [0, length)
        left_mean = (prefix[w:w+length] - prefix[:length]) / w
        # right_mean[p] = (prefix[p+2w] - prefix[p+w]) / w
        right_mean = (prefix[2*w:2*w+length] - prefix[w:w+length]) / w
        out[:, k] = (right_mean - left_mean).astype(np.float32)
    return out


def encode_triplet(seq_recomb, seq_parent1, seq_parent2):
    """Encode a triplet with one-hot + parent-comparison + MaxChi channels.

    Channels (shape (L, 22) after run #13):
        0-4    recombinant one-hot
        5-9    parent 1 one-hot
        10-14  parent 2 one-hot
        15     match_p1: recombinant base equals parent 1 base
        16     match_p2: recombinant base equals parent 2 base
        17     informative: parent 1 base differs from parent 2 base
        18-21  MaxChi-style right-minus-left disparity of (match_p1 - match_p2)
               at windows {50, 100, 200, 500} bp.

    All comparison and MaxChi channels are zero at padding positions (and
    at any position where the running window only sees padding), so they
    do not add spurious signal where there is no data.
    """
    enc_r = one_hot_encode(seq_recomb)
    enc_1 = one_hot_encode(seq_parent1)
    enc_2 = one_hot_encode(seq_parent2)

    r  = _seq_to_index(seq_recomb)
    p1 = _seq_to_index(seq_parent1)
    p2 = _seq_to_index(seq_parent2)
    valid = (r >= 0) & (p1 >= 0) & (p2 >= 0)

    match_p1   = ((r  == p1) & valid).astype(np.float32)
    match_p2   = ((r  == p2) & valid).astype(np.float32)
    informative = ((p1 != p2) & valid).astype(np.float32)

    parental_signal = (match_p1 - match_p2).astype(np.float32)
    maxchi = _maxchi_features(parental_signal)

    return np.concatenate(
        [enc_r, enc_1, enc_2,
         match_p1[:, None], match_p2[:, None], informative[:, None],
         maxchi],
        axis=1,
    )


# Sanity checks
test_enc = one_hot_encode("ATGC-N", max_length=6)
print(f"One-hot shape: {test_enc.shape}")
print(f"A=[1,0,0,0,0]: {test_enc[0].tolist()}")
print(f"T=[0,1,0,0,0]: {test_enc[1].tolist()}")
print(f"Gap=[-]:       {test_enc[4].tolist()}")
print(f"Unknown->gap:  {test_enc[5].tolist()}")

trip = encode_triplet("ATGCAT", "ATGGGT", "ATCCAT")
print(f"\nTriplet shape: {trip.shape}  (expected ({MAX_SEQ_LEN}, {N_INPUT_CHANNELS}))")
print(f"Pos 2 comparison [m_p1, m_p2, info]: {trip[2, 15:18].tolist()}")
print(f"Pos 3 comparison [m_p1, m_p2, info]: {trip[3, 15:18].tolist()}")
print(f"Pos 0 comparison [m_p1, m_p2, info]: {trip[0, 15:18].tolist()}")
print(f"Pos 2 MaxChi (4 windows):            {[round(x, 4) for x in trip[2, 18:22].tolist()]}")
print(f"Pos 3 MaxChi (4 windows):            {[round(x, 4) for x in trip[3, 18:22].tolist()]}")


One-hot shape: (6, 5)
A=[1,0,0,0,0]: [1.0, 0.0, 0.0, 0.0, 0.0]
T=[0,1,0,0,0]: [0.0, 1.0, 0.0, 0.0, 0.0]
Gap=[-]:       [0.0, 0.0, 0.0, 0.0, 1.0]
Unknown->gap:  [0.0, 0.0, 0.0, 0.0, 1.0]



Triplet shape: (32000, 22)  (expected (32000, 22))
Pos 2 comparison [m_p1, m_p2, info]: [1.0, 0.0, 1.0]
Pos 3 comparison [m_p1, m_p2, info]: [0.0, 1.0, 1.0]
Pos 0 comparison [m_p1, m_p2, info]: [1.0, 1.0, 0.0]
Pos 2 MaxChi (4 windows):            [-0.02, -0.01, -0.005, -0.002]
Pos 3 MaxChi (4 windows):            [-0.06, -0.03, -0.015, -0.006]


### Label generation

Per-position binary labels with two modes:

- **Breakpoint** (default): only positions within +/- `BP_WINDOW` (10 bp)
  of each breakpoint edge are positive. Trains the model to pinpoint exact
  breakpoint transitions.
- **Region**: every position between `SimBPStart` and `SimBPEnd` is positive.
  Captures the full recombinant region.

Circular genomes (where `bp_start > bp_end`, meaning the recombinant
region wraps around the genome end) are handled explicitly in `region` mode.
In `breakpoint` mode the two breakpoint windows are independent, so
circularity is automatic.

In [5]:
def generate_labels(bp_start, bp_end, seq_length=MAX_SEQ_LEN,
                    mode=None, window=BP_WINDOW, sigma=LABEL_SIGMA):
    """Generate per-position labels from breakpoint coordinates.

    Modes:
        'gaussian'   -- soft target: each breakpoint contributes a
                        Gaussian peak of width `sigma`. Smooth gradient
                        in [0, 1], aligned with peak-based evaluation.
        'breakpoint' -- hard +/- `window` bp binary edges.
        'region'     -- entire recombinant span between breakpoints.

    Args:
        bp_start, bp_end: Breakpoint coordinates.
        seq_length: Length of the label vector.
        mode: Defaults to LABEL_MODE if not given.
        window: Half-width for 'breakpoint' mode.
        sigma:  Gaussian half-width for 'gaussian' mode.

    Returns:
        np.ndarray of shape (seq_length,), float32.
    """
    if mode is None:
        mode = LABEL_MODE

    labels = np.zeros(seq_length, dtype=np.float32)
    circular = bp_start > bp_end  # Circular genome wrapping

    if mode == 'gaussian':
        # Each breakpoint is its own peak; circular wrap handled
        # implicitly because each bp is placed at its own coord.
        pos = np.arange(seq_length, dtype=np.float32)
        for bp in [bp_start, bp_end]:
            peak = np.exp(-0.5 * ((pos - bp) / sigma) ** 2)
            labels = np.maximum(labels, peak.astype(np.float32))
    elif mode == 'region':
        if circular:
            labels[bp_start:] = 1.0
            labels[:bp_end + 1] = 1.0
        else:
            labels[bp_start:bp_end + 1] = 1.0
    elif mode == 'breakpoint':
        for bp in [bp_start, bp_end]:
            lo = max(0, bp - window)
            hi = min(seq_length, bp + window + 1)
            labels[lo:hi] = 1.0
    else:
        raise ValueError(f"Unknown label mode: {mode}")

    return labels


# Verify all modes
lbl_g  = generate_labels(500, 1500, mode='gaussian')
lbl_bp = generate_labels(500, 1500, mode='breakpoint')
lbl_rg = generate_labels(500, 1500, mode='region')
lbl_circ_g = generate_labels(3500, 200, mode='gaussian')

print(f"Gaussian   -- max={lbl_g.max():.3f}, sum={lbl_g.sum():.1f}, "
      f"argmaxes near (500, 1500): {np.argmax(lbl_g[400:600]) + 400}, "
      f"{np.argmax(lbl_g[1400:1600]) + 1400}")
print(f"Breakpoint positives:  {int(lbl_bp.sum())}")
print(f"Region positives:      {int(lbl_rg.sum())}")
print(f"Circular gaussian peaks at: "
      f"{np.argmax(lbl_circ_g[3400:3600]) + 3400}, "
      f"{np.argmax(lbl_circ_g[100:300]) + 100}")


Gaussian   -- max=1.000, sum=100.3, argmaxes near (500, 1500): 500, 1500
Breakpoint positives:  42
Region positives:      1001
Circular gaussian peaks at: 3500, 200


### Parsing one SANTA simulation

For every `.fa` file the parser:

1. Reads the FASTA into a `{seq_id: sequence}` dict.
2. Loads the two associated CSVs.
3. For each event in `SimVSRealCompare.csv`:
   - Identifies the recombinant by matching `ActualRecomb` against `ISeqs(A)`.
   - Picks one parent ID from each of the two non-recombinant rows.
   - Encodes the triplet and generates labels.
4. Returns a list of dicts ready for batching.

Events are silently skipped if any sequence is missing or the stats CSV
does not contain exactly 3 hypothesis rows.

In [6]:
def parse_simulation(fasta_path):
    """Parse one SANTA simulation run (FASTA + associated CSVs).

    For each recombination event the function extracts the recombinant
    sequence and two parent sequences, encodes the triplet, and
    generates per-position labels.

    Args:
        fasta_path: Path to the .fa alignment file.

    Returns:
        List of dicts with keys 'input', 'labels_gaussian', 'labels_bp',
        'labels_region', 'meta'.
    """
    fasta_path = Path(fasta_path)
    sim_csv = fasta_path.parent / f"{fasta_path.name}SimVSRealCompare.csv"
    stats_csv = fasta_path.parent / f"{fasta_path.name}RecombIdentifyStats.csv"

    if not sim_csv.exists() or not stats_csv.exists():
        return []

    # FASTA IDs are integers in this dataset
    try:
        seqs = {int(r.id): str(r.seq) for r in SeqIO.parse(fasta_path, 'fasta')}
    except Exception:
        return []

    try:
        sim = pd.read_csv(sim_csv, skipinitialspace=True)
        stats = pd.read_csv(stats_csv, skipinitialspace=True)
    except Exception:
        return []

    results = []
    for _, row in sim.iterrows():
        event = row['RDPEvent']
        recomb_id = int(row['ActualRecomb'])
        bp_start = int(row['SimBPStart'])
        bp_end = int(row['SimBPEnd'])

        # Each event has 3 hypothesis rows in the stats CSV. Skip events
        # that don't follow this structure (data corruption, partial run).
        ev_rows = stats[stats['Event'] == event]
        if len(ev_rows) != 3:
            continue

        # Identify parents: the two hypothesis rows whose ISeqs(A) does
        # NOT contain the actual recombinant ID. ISeqs(A) is a $-delimited
        # list of seq IDs; we take the first valid integer from each.
        parent_ids = []
        for _, sr in ev_rows.iterrows():
            ids = [int(s.strip()) for s in str(sr['ISeqs(A)']).split('$')
                   if s.strip().isdigit()]
            if recomb_id in ids:
                continue
            if ids:
                parent_ids.append(ids[0])

        if len(parent_ids) < 2:
            continue

        if not all(sid in seqs for sid in [recomb_id, parent_ids[0], parent_ids[1]]):
            continue

        triplet = encode_triplet(
            seqs[recomb_id], seqs[parent_ids[0]], seqs[parent_ids[1]]
        )

        results.append({
            'input': triplet,
            'labels_gaussian': generate_labels(bp_start, bp_end, mode='gaussian'),
            'labels_bp': generate_labels(bp_start, bp_end, mode='breakpoint'),
            'labels_region': generate_labels(bp_start, bp_end, mode='region'),
            'meta': {
                'file': fasta_path.name,
                'event': event,
                'recomb_id': recomb_id,
                'parent1_id': parent_ids[0],
                'parent2_id': parent_ids[1],
                'bp_start': bp_start,
                'bp_end': bp_end,
                # rstrip trailing gaps; internal gaps are kept (they are
                # alignment artefacts and the breakpoint coordinates are
                # in alignment space, not raw-sequence space).
                'actual_len': len(seqs[recomb_id].rstrip('-')),
            },
        })

    return results


# Smoke test on one file
sample_fa = sorted((DATA_ROOT / "XML-1").glob("*.fa"))[0]
sample_triplets = parse_simulation(sample_fa)
print(f"Parsed {len(sample_triplets)} triplets from {sample_fa.name}")
if sample_triplets:
    print(f"Input shape:  {sample_triplets[0]['input'].shape}")
    print(f"Labels shape: {sample_triplets[0]['labels_gaussian'].shape}")
    print(f"Labels max (gaussian): {sample_triplets[0]['labels_gaussian'].max():.3f}")
    print(f"Metadata:     {sample_triplets[0]['meta']}")


Parsed 0 triplets from alignment_XML1-2500-0.005-12E-5-100-1.fa


In [7]:
# Disk cache for parsed datasets. The MaxChi computation in
# encode_triplet is the bottleneck of data prep (per-file ~tens of ms;
# at full XML-1..4 it's the dominant wall-time cost on a single CPU).
# We cache the full per-directory `(X, y_gaussian, y_breakpoint,
# y_region, mask)` and the meta list once parsed, keyed on the config
# knobs that change the encoding/labels.
#
# Cache invalidates automatically when:
#   - MAX_SEQ_LEN changes (encoded shape changes)
#   - MAXCHI_WINDOWS changes (channel count + values change)
#   - LABEL_SIGMA changes (Gaussian labels change)
#   - BP_WINDOW changes (breakpoint-mode labels change)
#   - the file list (sorted) under DATA_ROOT/<dir> changes
#   - max_files truncation differs
#
# Cache invalidates manually by bumping CACHE_VERSION below.
import hashlib, pickle

CACHE_DIR = Path('cache')
CACHE_DIR.mkdir(exist_ok=True)
CACHE_VERSION = 'v1'  # bump when encode_triplet / generate_labels semantics change


def _dataset_cache_key(directory_name, fa_files, max_files):
    payload = (
        CACHE_VERSION,
        directory_name,
        tuple(f.name for f in fa_files),
        int(MAX_SEQ_LEN),
        int(N_INPUT_CHANNELS),
        tuple(MAXCHI_WINDOWS),
        float(LABEL_SIGMA),
        int(BP_WINDOW),
        max_files if max_files is not None else 'all',
    )
    return hashlib.sha256(repr(payload).encode()).hexdigest()[:16]


def load_dataset(directories, label_mode=None, max_files=None):
    """Load and aggregate triplet data from multiple simulation directories.

    Args:
        directories: List of subdirectory names under DATA_ROOT.
        label_mode: 'gaussian' | 'breakpoint' | 'region'. Defaults to LABEL_MODE.
        max_files: Optional per-directory file limit (for quick testing).

    Returns:
        X    -- np.ndarray (n, MAX_SEQ_LEN, N_INPUT_CHANNELS)
        y    -- np.ndarray (n, MAX_SEQ_LEN)  (selected by label_mode)
        mask -- np.ndarray (n, MAX_SEQ_LEN)  per-position validity (1.0 = real, 0.0 = padding)
        meta -- list of metadata dicts

    Caching: per-directory results are cached to disk under `cache/`.
    """
    if label_mode is None:
        label_mode = LABEL_MODE
    label_keys = {
        'gaussian':   'labels_gaussian',
        'breakpoint': 'labels_bp',
        'region':     'labels_region',
    }
    if label_mode not in label_keys:
        raise ValueError(f"Unknown label_mode: {label_mode}")

    inputs, labels_g, labels_bp, labels_rg, meta = [], [], [], [], []

    for d in directories:
        fa_files = sorted((DATA_ROOT / d).glob("*.fa"))
        if max_files:
            fa_files = fa_files[:max_files]

        cache_key = _dataset_cache_key(d, fa_files, max_files)
        npz_path = CACHE_DIR / f"ds_{d}_{cache_key}.npz"
        pkl_path = CACHE_DIR / f"ds_{d}_{cache_key}.pkl"

        if npz_path.exists() and pkl_path.exists():
            print(f"\nCache HIT: {d}  ({npz_path.name}, {npz_path.stat().st_size/1e6:.1f} MB)")
            data = np.load(npz_path)
            X_d   = data['X']
            yg_d  = data['y_g']
            ybp_d = data['y_bp']
            yrg_d = data['y_rg']
            with open(pkl_path, 'rb') as f:
                meta_d = pickle.load(f)
            print(f"  Loaded {X_d.shape[0]} samples from cache")
        else:
            print(f"\nCache MISS: {d}  ({len(fa_files)} files; will parse and cache)")
            X_l, yg_l, ybp_l, yrg_l, meta_l = [], [], [], [], []
            for fa in tqdm(fa_files, desc=d):
                for t in parse_simulation(fa):
                    X_l.append(t['input'])
                    yg_l.append(t['labels_gaussian'])
                    ybp_l.append(t['labels_bp'])
                    yrg_l.append(t['labels_region'])
                    meta_l.append(t['meta'])
            X_d   = np.array(X_l,   dtype=np.float32) if X_l   else np.zeros((0, MAX_SEQ_LEN, N_INPUT_CHANNELS), dtype=np.float32)
            yg_d  = np.array(yg_l,  dtype=np.float32) if yg_l  else np.zeros((0, MAX_SEQ_LEN), dtype=np.float32)
            ybp_d = np.array(ybp_l, dtype=np.float32) if ybp_l else np.zeros((0, MAX_SEQ_LEN), dtype=np.float32)
            yrg_d = np.array(yrg_l, dtype=np.float32) if yrg_l else np.zeros((0, MAX_SEQ_LEN), dtype=np.float32)
            meta_d = meta_l
            print(f"  Parsed {X_d.shape[0]} samples; saving cache ...")
            np.savez_compressed(npz_path, X=X_d, y_g=yg_d, y_bp=ybp_d, y_rg=yrg_d)
            with open(pkl_path, 'wb') as f:
                pickle.dump(meta_d, f)
            print(f"  Cache written: {npz_path.name} ({npz_path.stat().st_size/1e6:.1f} MB)")

        inputs.append(X_d)
        labels_g.append(yg_d)
        labels_bp.append(ybp_d)
        labels_rg.append(yrg_d)
        meta.extend(meta_d)

    X = np.concatenate(inputs, axis=0) if inputs else np.zeros((0, MAX_SEQ_LEN, N_INPUT_CHANNELS), dtype=np.float32)
    if label_mode == 'gaussian':
        y = np.concatenate(labels_g, axis=0) if labels_g else np.zeros((0, MAX_SEQ_LEN), dtype=np.float32)
    elif label_mode == 'breakpoint':
        y = np.concatenate(labels_bp, axis=0) if labels_bp else np.zeros((0, MAX_SEQ_LEN), dtype=np.float32)
    else:  # region
        y = np.concatenate(labels_rg, axis=0) if labels_rg else np.zeros((0, MAX_SEQ_LEN), dtype=np.float32)

    # Padding past every sequence end has all channels equal to zero.
    mask = (X.sum(axis=-1) > 0).astype(np.float32)
    pos_mass = float(np.sum(y))
    valid_frac = float(mask.mean()) if mask.size else 0.0
    print(f"\n{'='*60}")
    print(f"Loaded {X.shape[0]} samples (label_mode={label_mode})")
    print(f"X shape: {X.shape}  |  y shape: {y.shape}  |  mask shape: {mask.shape}")
    if y.size:
        print(f"Label mass (sum y): {pos_mass:.1f} / {y.size}  "
              f"(mean={y.mean():.5f}, max={y.max():.3f})")
    print(f"Valid (non-padded) fraction: {valid_frac:.3f}")
    print(f"{'='*60}")

    return X, y, mask, meta


In [8]:
# Load training and validation data with a STRICT held-out split:
# XML-1..4 → train, XML-5 → val.
# Per run #14: when training on all 5 XMLs and shuffling by file, val
# (which is mixed across XMLs) tracks train closely, and val gains
# don't transfer to test. Holding out an entire XML directory tests
# cross-configuration generalization — closer to the simulation→real
# transfer the project ultimately wants.
TRAIN_DIRS_INNER = ["XML-1", "XML-2", "XML-3", "XML-4"]
VAL_DIRS_INNER = ["XML-5"]

print("=== TRAIN: XML-1..4 ===")
# Run #24: max_files capped at 500 per train dir.  At MAX_SEQ_LEN=32000
# the full-set load (~3331 events) panics the 15 GB WSL VM during
# np.concatenate inside load_dataset (peaks ~18 GB).  Run #14 evidence
# shows full-set vs 750 was +0.015 val F1 (noise) — capping at 500 (~1780
# events) is plenty for testing the truncation hypothesis.  Val (XML-5)
# is unchanged so it stays comparable to runs #15-#23.  If #24 lands
# KEPT, run #25 should engineer the loader (incremental concat, float16)
# and re-test at full data.
X_train_all, y_train_all, mask_train_all, meta_train_all = load_dataset(
    TRAIN_DIRS_INNER, max_files=500,
)

print("\n=== VAL: XML-5 ===")
X_val_all, y_val_all, mask_val_all, meta_val_all = load_dataset(
    VAL_DIRS_INNER, max_files=None,
)

# POS_WEIGHT computed from TRAIN ONLY (val isn't visible at training time).
mean_y_unmasked = float((y_train_all * mask_train_all).sum() / mask_train_all.sum())
implied_pos_weight = float((1.0 - mean_y_unmasked) / mean_y_unmasked)
# Run #24: POS_WEIGHT rescaled 70 → 178 paired with MAX_SEQ_LEN 10k → 32k.
# Sequences ~3× longer at same positives-per-event (2 bps), so mean(y)
# falls ~3× and data-implied pos_weight triples. Run #8 found data-implied
# is over-aggressive at σ=20; apply same 70/82.62 = 0.847× factor:
# 210 × 0.847 ≈ 178. If #24 KEPT, this becomes the new baseline.
POS_WEIGHT = 178.0
print(f"\n{'='*60}")
print(f"TRAIN: {X_train_all.shape[0]} samples; mean(y) over unmasked: {mean_y_unmasked:.5f}")
print(f"VAL:   {X_val_all.shape[0]} samples")
print(f"Implied POS_WEIGHT (from train data): {implied_pos_weight:.2f}")
print(f"POS_WEIGHT in use (hardcoded):        {POS_WEIGHT:.2f}")
print(f"{'='*60}")

# Run #24: skipped the X_all/y_all/mask_all concatenation. At
# MAX_SEQ_LEN=32000 the concat would peak ~11 GB on top of the existing
# train/val arrays (15 GB system RAM). The variables were only used by
# legacy saliency cells that are no longer present — searching the
# notebook turned up no live consumer. Re-add the concat (or compute
# lazily) if a downstream cell needs the unified view.
import numpy as np  # noqa: F401  (kept for downstream cells)


=== TRAIN: XML-1..4 ===



Cache HIT: XML-1  (ds_XML-1_8081e50960addb6c.npz, 11.2 MB)


  Loaded 338 samples from cache

Cache HIT: XML-2  (ds_XML-2_7c8311d89a5788d7.npz, 25.5 MB)


  Loaded 422 samples from cache

Cache HIT: XML-3  (ds_XML-3_8b07772683bedab2.npz, 42.1 MB)


  Loaded 890 samples from cache

Cache HIT: XML-4  (ds_XML-4_0a567a79ba076738.npz, 9.1 MB)


...


Loaded 1748 samples (label_mode=gaussian)
X shape: (1748, 32000, 22)  |  y shape: (1748, 32000)  |  mask shape: (1748, 32000)
Label mass (sum y): 164387.3 / 55936000  (mean=0.00294, max=1.000)
Valid (non-padded) fraction: 0.224



=== VAL: XML-5 ===

Cache HIT: XML-5  (ds_XML-5_8c9b9e1de5649477.npz, 42.5 MB)


  Loaded 621 samples from cache



Loaded 621 samples (label_mode=gaussian)
X shape: (621, 32000, 22)  |  y shape: (621, 32000)  |  mask shape: (621, 32000)
Label mass (sum y): 57806.4 / 19872000  (mean=0.00291, max=1.000)
Valid (non-padded) fraction: 0.343



TRAIN: 1748 samples; mean(y) over unmasked: 0.01246
VAL:   621 samples
Implied POS_WEIGHT (from train data): 79.25
POS_WEIGHT in use (hardcoded):        178.00


In [9]:
# Train / validation split — DIRECTORY-LEVEL HELD OUT (run #15+).
# XML-1..4 are train, XML-5 is val. No file-level shuffling needed since
# the split is already directory-based. Variables are renamed to match
# the conventions downstream cells expect (X_train, X_val, w_train, w_val).
X_train = X_train_all
y_train = y_train_all
w_train = mask_train_all
meta_train = meta_train_all

X_val = X_val_all
y_val = y_val_all
w_val = mask_val_all
meta_val = meta_val_all

print(f"Held-out split: XML-1..4 (train) | XML-5 (val)")
print(f"Training:   {X_train.shape[0]} samples  (valid positions: {w_train.mean():.3f})")
print(f"Validation: {X_val.shape[0]} samples  (valid positions: {w_val.mean():.3f})")


Held-out split: XML-1..4 (train) | XML-5 (val)
Training:   1748 samples  (valid positions: 0.224)
Validation: 621 samples  (valid positions: 0.343)


## Loss function and evaluation metrics

### Focal loss for class imbalance

Breakpoint positions are extremely rare (<1% of all positions), so
plain binary cross-entropy is dominated by the negative class. **Focal
loss** down-weights easy negatives by `(1 - p_t)^gamma`, focusing
gradient updates on the rare positives the model still gets wrong.

### Peak-based tolerance metric

A predicted breakpoint counts as correct if some peak in the model
output lands within +/- `TOLERANCE` (200 bp) of a true breakpoint.
We use `scipy.signal.find_peaks` with `min_distance=TOLERANCE` so
broad high-probability ridges collapse into a single predicted
breakpoint -- otherwise one wide bump would be counted as dozens of
true positives. Each true breakpoint can only match one peak (greedy
nearest-first), giving honest precision and recall numbers against the
two real breakpoints per sample.

In [10]:
# ----- LEGACY per-position losses (kept for reference / A-B switching back) -----

def focal_loss(alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA):
    """Focal loss for extreme class imbalance (LEGACY). See cell-15 history."""
    def loss(y_true, y_pred):
        eps = K.epsilon()
        y_pred = K.clip(y_pred, eps, 1.0 - eps)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        p_t     = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        focal_w = K.pow(1 - p_t, gamma)
        bce = -(y_true * K.log(y_pred) + (1 - y_true) * K.log(1 - y_pred))
        return K.mean(alpha_t * focal_w * bce)
    return loss


def weighted_bce(pos_weight=POS_WEIGHT):
    """Class-weighted BCE for soft Gaussian targets (LEGACY)."""
    def loss(y_true, y_pred):
        eps = K.epsilon()
        y_pred = K.clip(y_pred, eps, 1.0 - eps)
        weight = pos_weight * y_true + (1.0 - y_true)
        per_pos = -(y_true * K.log(y_pred) + (1 - y_true) * K.log(1 - y_pred))
        return K.mean(weight * per_pos)
    return loss


# ----- TOP-K coordinate regression loss + metric (active path, run #19+) -----

def topk_xent_loss(y_true, y_pred):
    """Categorical cross-entropy across K independent softmax heads.

    Args:
        y_true: (batch, K, L) one-hot at sorted true bp positions.
        y_pred: (batch, K, L) softmax over positions per head.

    Returns:
        Scalar loss = mean over batch and K of -sum_pos[y_true * log(y_pred)].
        For one-hot targets this reduces to -log(y_pred at the true position),
        averaged. Range [0, +inf); 0 = perfect concentration on truth.
    """
    eps = 1e-9
    # (batch, K) per-head per-sample cross-entropy
    per_head = -tf.reduce_sum(y_true * tf.math.log(y_pred + eps), axis=-1)
    return tf.reduce_mean(per_head)


def topk_match_rate(y_true, y_pred):
    """Fraction of (sample, head) pairs where argmax falls within ±TOLERANCE
    of the head's true position.

    With K=2 sorted heads, this is "fraction of the 2N predictions that
    are correct" — a softer signal than "Both BPs found", which requires
    BOTH heads correct on the same sample. Used for monitoring / early
    stopping; the headline metric (Both BPs found %) is computed at eval
    time on the val/test sets.
    """
    true_pos = tf.argmax(y_true, axis=-1, output_type=tf.int32)  # (batch, K)
    pred_pos = tf.argmax(y_pred, axis=-1, output_type=tf.int32)  # (batch, K)
    within = tf.abs(true_pos - pred_pos) <= TOLERANCE
    return tf.reduce_mean(tf.cast(within, tf.float32))


In [11]:
# ----- LEGACY per-position peak detection (kept for reference) -----

def detect_peaks(y_pred, threshold=0.5, min_distance=TOLERANCE):
    """Detect breakpoint peaks in a per-position prediction vector (LEGACY)."""
    peaks, _ = find_peaks(y_pred, height=threshold, distance=min_distance)
    return peaks


def peak_metrics_single(true_bps, y_pred, tolerance=TOLERANCE,
                        threshold=0.5, min_distance=None):
    """Peak-based precision / recall / F1 for a single sample (LEGACY)."""
    if min_distance is None:
        min_distance = tolerance
    peaks = detect_peaks(y_pred, threshold=threshold, min_distance=min_distance)
    true_bps = list(true_bps)
    if len(peaks) == 0 and len(true_bps) == 0:
        return 1.0, 1.0, 1.0, 0, 0
    if len(peaks) == 0:
        return 0.0, 0.0, 0.0, 0, 0
    if len(true_bps) == 0:
        return 0.0, 0.0, 0.0, len(peaks), 0
    pairs = sorted(((abs(int(p) - int(t)), pi, ti)
                    for pi, p in enumerate(peaks)
                    for ti, t in enumerate(true_bps)),
                   key=lambda x: x[0])
    matched_pred, matched_true = set(), set()
    for d, pi, ti in pairs:
        if d > tolerance: break
        if pi in matched_pred or ti in matched_true: continue
        matched_pred.add(pi); matched_true.add(ti)
    tp = len(matched_true)
    fp = len(peaks) - tp
    fn = len(true_bps) - tp
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f, len(peaks), tp


def evaluate_peaks(y_pred_batch, meta_batch, tolerance=TOLERANCE, threshold=0.5):
    """Aggregate peak-based metrics across a batch of samples (LEGACY)."""
    total_tp, total_fp, total_fn = 0, 0, 0
    for yp, m in zip(y_pred_batch, meta_batch):
        true_bps = [m['bp_start'], m['bp_end']]
        peaks = detect_peaks(yp, threshold=threshold)
        if len(peaks) == 0 and len(true_bps) == 0: continue
        if len(peaks) == 0:
            total_fn += len(true_bps); continue
        if len(true_bps) == 0:
            total_fp += len(peaks); continue
        pairs = sorted(((abs(int(p) - int(t)), pi, ti)
                        for pi, p in enumerate(peaks)
                        for ti, t in enumerate(true_bps)),
                       key=lambda x: x[0])
        matched_pred, matched_true = set(), set()
        for d, pi, ti in pairs:
            if d > tolerance: break
            if pi in matched_pred or ti in matched_true: continue
            matched_pred.add(pi); matched_true.add(ti)
        tp = len(matched_true)
        total_tp += tp
        total_fp += len(peaks) - tp
        total_fn += len(true_bps) - tp
    p = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    r = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    f = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f


# ----- TOP-K eval helpers (active path, run #19+) -----

def make_topk_targets(meta_list, max_len=None, K=None, sigma=None):
    """Build (n, K, max_len) targets from per-event meta.

    For each event: sort (bp_start, bp_end) ascending. Per head k, place
    a Gaussian peak (normalized to sum to 1) centered at the sorted bp.
    With sigma=0 the target is a hard one-hot at the bp (run #19).
    With sigma>0 the target is a soft Gaussian (run #20+, σ=5 default)
    that spreads mass over ~6σ neighboring positions, giving a smoother
    cross-entropy gradient than one-hot while still encouraging sharp
    argmax.

    Targets are sorted ascending so head-0 always learns the *earlier*
    breakpoint and head-1 the *later* — gives the model a stable
    learning signal under K=2 sorted-pair matching.

    Out-of-range positions are clamped to [0, max_len-1] (rare; SANTA bp
    coords usually fall within MAX_SEQ_LEN).
    """
    if max_len is None: max_len = MAX_SEQ_LEN
    if K is None: K = K_TOPK
    if sigma is None: sigma = TOPK_TARGET_SIGMA  # cell-3 constant
    edge = EDGE_BUFFER  # also defined in cell-3
    n = len(meta_list)
    targets = np.zeros((n, K, max_len), dtype=np.float32)
    pos = np.arange(max_len, dtype=np.float32)
    n_skipped = 0
    for i, m in enumerate(meta_list):
        bps = sorted([int(m['bp_start']), int(m['bp_end'])])
        for k in range(K):
            if k < len(bps):
                bp = bps[k]
                # Skip out-of-encoded-range bps (bp_end can exceed MAX_SEQ_LEN
                # in the SANTA data) AND bps inside the edge-suppression buffer
                # (the head can't predict there anyway, so forcing a target
                # would create a constant cross-entropy penalty without useful
                # gradient). Setting target = all-zeros means cross-entropy
                # contributes 0 loss / 0 gradient for this (sample, head).
                if bp < edge or bp >= max_len - edge:
                    n_skipped += 1
                    continue
                if sigma <= 0:
                    targets[i, k, bp] = 1.0
                else:
                    g = np.exp(-0.5 * ((pos - bp) / sigma) ** 2)
                    g /= g.sum()  # normalise to sum to 1 so it's a valid distribution
                    targets[i, k] = g.astype(np.float32)
    if n_skipped > 0:
        print(f"  make_topk_targets: skipped {n_skipped} (sample, head) pairs "
              f"with bp out of [{edge}, {max_len - edge}) — set to all-zero")
    return targets


def topk_predict_positions(y_pred):
    """Argmax of each head, sorted ascending for sorted-pair matching.

    Args:
        y_pred: (n, K, L) softmax distributions.

    Returns:
        pred_pos: (n, K) integer positions, sorted ascending per row.
    """
    pred_pos = np.argmax(y_pred, axis=-1)
    return np.sort(pred_pos, axis=-1)


def topk_event_metrics(y_pred, meta_list, tolerance=TOLERANCE, K=None):
    """Top-K event-level eval. Same DataFrame columns as legacy
    build_event_df so downstream length/circular breakdowns still work.

    Returns:
        events_df: per-event status (both/one/missed), distances, predictions.
        peak_p, peak_r, peak_f: precision/recall/F1 against ±tolerance match.
    """
    if K is None: K = K_TOPK
    pred_pos = topk_predict_positions(y_pred)  # (n, K) sorted
    rows = []
    total_tp, total_fp, total_fn = 0, 0, 0
    for i, m in enumerate(meta_list):
        true_bps = sorted([int(m['bp_start']), int(m['bp_end'])])
        preds = list(pred_pos[i])
        unmatched = list(preds)
        distances = [np.nan] * len(true_bps)
        for ti, tb in enumerate(true_bps):
            if not unmatched: break
            best_j = min(range(len(unmatched)),
                         key=lambda j: abs(unmatched[j] - tb))
            d = abs(unmatched[best_j] - tb)
            if d <= tolerance:
                distances[ti] = int(d)
                unmatched.pop(best_j)
        n_matched = sum(1 for d in distances if not np.isnan(d))
        if n_matched == len(true_bps):
            status = 'both'
        elif n_matched >= 1:
            status = 'one'
        else:
            status = 'missed'
        rows.append({
            'file': m['file'],
            'event': m['event'],
            'actual_len': m['actual_len'],
            'circular': m['bp_start'] > m['bp_end'],
            'status': status,
            'n_matched': n_matched,
            'd_start': distances[0],
            'd_end':   distances[1] if len(distances) > 1 else np.nan,
            'n_peaks': K,                     # always K under top-K
            'n_extra': K - n_matched,
            'pred_bp1': int(preds[0]),
            'pred_bp2': int(preds[1]) if len(preds) > 1 else -1,
        })
        total_tp += n_matched
        total_fn += len(true_bps) - n_matched
        total_fp += K - n_matched
    P = total_tp / (total_tp + total_fp) if (total_tp + total_fp) else 0.0
    R = total_tp / (total_tp + total_fn) if (total_tp + total_fn) else 0.0
    F = 2 * P * R / (P + R) if (P + R) else 0.0
    return pd.DataFrame(rows), P, R, F


def print_topk_event_summary(events_df, title):
    """Same shape as the legacy print_event_summary so log entries align."""
    n_total = len(events_df)
    n_both   = int((events_df["status"] == "both").sum())
    n_one    = int((events_df["status"] == "one").sum())
    n_missed = int((events_df["status"] == "missed").sum())
    print("=" * 60)
    print(title)
    print("=" * 60)
    print(f"Total events:            {n_total}")
    print(f"Both BPs found:          {n_both:4d}  ({n_both/n_total:.1%})")
    print(f"One BP found (partial):  {n_one:4d}  ({n_one/n_total:.1%})")
    print(f"Missed entirely:         {n_missed:4d}  ({n_missed/n_total:.1%})")
    print(f"Any BP found (>=1):      {n_both + n_one:4d}  ({(n_both + n_one)/n_total:.1%})")
    print()
    print(f"Mean predicted positions per event: {events_df['n_peaks'].mean():.2f} (always K under top-K)")
    print(f"Mean false-positive predictions:    {events_df['n_extra'].mean():.2f}")


## Multi-scale 1D CNN

**Idea**: different kernel sizes capture features at different scales.
Short kernels (3, 7) latch onto local nucleotide flips; longer kernels
(15, 31) pick up compositional shifts that span tens of bases.

**Architecture in one diagram**:

```
input (10000, 22)
        |
   shared Conv k=3 -> BN -> ReLU
        |
   +----+----+----+----+
   |    |    |    |    |
  k=3  k=7  k=15 k=31    (each: Conv -> BN -> ReLU -> Dropout)
   |    |    |    |
   +----+----+----+----+
        |
   concatenate
        |
   residual dilated stack (k=7, d=1..32)
        |
   Conv k=1 -> ReLU -> Dropout
   Conv k=1 -> sigmoid (with prior-aware bias init)
        |
   output (10000,)
```

Every conv uses `padding='same'`, so the output length matches the
input length and we get a probability per position. Compile with
`weighted_bce(POS_WEIGHT)` and `tf.keras.metrics.AUC(curve='PR')`.

(Top-K axis explored in runs #19-#21 — REVERTED, see Experiment Log.
Reverted to per-position pipeline at run #22.)


In [12]:
import math


# Per-position build_cnn (run #16 baseline; restored at run #22 after top-K REVERTED)
def build_cnn(input_shape=(MAX_SEQ_LEN, N_INPUT_CHANNELS),
              filter_sizes=(3, 7, 15, 31),  # Run #22 tested (7,31,63,127) — REVERTED
              dilation_rates=(1, 2, 4, 8, 16, 32),
              n_filters=64,
              dropout=0.5,
              prior_positive=0.01):
    """Multi-scale 1D CNN with WaveNet-style residual dilated stack.

    Architecture (run #16 baseline — restored after top-K runs #19-#21 REVERTED):
        1. Shared initial convolution.
        2. Parallel multi-scale branches at the input layer.
        3. Stack of dilated convolutions wrapped in residual blocks.
        4. Position-wise classification head with prior-aware bias init.

    Returns Keras Model with output shape (batch, MAX_SEQ_LEN) — sigmoid probabilities.
    """
    inputs = Input(shape=input_shape, name='triplet_input')
    x = Conv1D(64, 3, padding='same', name='conv_shared')(inputs)
    x = BatchNormalization(name='bn_shared')(x)
    x = Activation('relu', name='relu_shared')(x)

    branches = []
    for ks in filter_sizes:
        b = Conv1D(n_filters, ks, padding='same', name=f'conv_k{ks}')(x)
        b = BatchNormalization(name=f'bn_k{ks}')(b)
        b = Activation('relu', name=f'relu_k{ks}')(b)
        b = Dropout(dropout, name=f'drop_k{ks}')(b)
        branches.append(b)
    x = Concatenate(name='merge_scales')(branches)

    for i, d in enumerate(dilation_rates):
        if i == 0:
            residual = Conv1D(n_filters, 1, padding='same', name='res_proj_d1')(x)
        else:
            residual = x
        y = Conv1D(n_filters, 7, padding='same', dilation_rate=d, name=f'conv_dil_d{d}')(x)
        y = BatchNormalization(name=f'bn_dil_d{d}')(y)
        y = Activation('relu', name=f'relu_dil_d{d}')(y)
        y = Dropout(dropout, name=f'drop_dil_d{d}')(y)
        x = Add(name=f'res_add_d{d}')([y, residual])

    x = Conv1D(32, 1, name='conv_pw1')(x)
    x = Activation('relu', name='relu_pw1')(x)
    x = Dropout(dropout, name='drop_pw')(x)
    bias_init = -math.log((1 - prior_positive) / prior_positive)
    x = Conv1D(1, 1, activation='sigmoid',
               bias_initializer=tf.keras.initializers.Constant(bias_init),
               name='conv_out')(x)
    outputs = Reshape((MAX_SEQ_LEN,), name='output')(x)
    return Model(inputs, outputs, name='MultiScaleCNN_PerPosition')


# (build_cnn_topk is still defined further up in this cell for historical reference,
# but per-position cnn is the active model from run #22 onward.)
def build_cnn_topk(input_shape=(MAX_SEQ_LEN, N_INPUT_CHANNELS),
                    filter_sizes=(3, 7, 15, 31),
                    dilation_rates=(1, 2, 4, 8, 16, 32),
                    n_filters=64, dropout=0.5, K=None):
    """LEGACY top-K head — REVERTED at run #22 (axis closed, see Experiment Log #21)."""
    if K is None: K = K_TOPK
    inputs = Input(shape=input_shape, name='triplet_input')
    valid_mask = Lambda(
        lambda t: tf.reduce_max(tf.cast(tf.not_equal(t, 0), tf.float32),
                                axis=-1, keepdims=True),
        name='compute_valid_mask',
    )(inputs)
    x = Conv1D(64, 3, padding='same', name='conv_shared')(inputs)
    x = BatchNormalization(name='bn_shared')(x)
    x = Activation('relu', name='relu_shared')(x)
    branches = []
    for ks in filter_sizes:
        b = Conv1D(n_filters, ks, padding='same', name=f'conv_k{ks}')(x)
        b = BatchNormalization(name=f'bn_k{ks}')(b)
        b = Activation('relu', name=f'relu_k{ks}')(b)
        b = Dropout(dropout, name=f'drop_k{ks}')(b)
        branches.append(b)
    x = Concatenate(name='merge_scales')(branches)
    for i, d in enumerate(dilation_rates):
        if i == 0:
            residual = Conv1D(n_filters, 1, padding='same', name='res_proj_d1')(x)
        else:
            residual = x
        y = Conv1D(n_filters, 7, padding='same', dilation_rate=d, name=f'conv_dil_d{d}')(x)
        y = BatchNormalization(name=f'bn_dil_d{d}')(y)
        y = Activation('relu', name=f'relu_dil_d{d}')(y)
        y = Dropout(dropout, name=f'drop_dil_d{d}')(y)
        x = Add(name=f'res_add_d{d}')([y, residual])
    x = Conv1D(32, 1, name='conv_pw1')(x)
    x = Activation('relu', name='relu_pw1')(x)
    x = Dropout(dropout, name='drop_pw')(x)
    logits = Conv1D(K, 1, padding='same', name='conv_logits_k')(x)
    def _mask_pad_edge(args):
        L = MAX_SEQ_LEN
        idx = tf.range(L)
        interior = tf.cast(tf.logical_and(idx >= EDGE_BUFFER, idx < L - EDGE_BUFFER), tf.float32)
        interior = tf.reshape(interior, (1, L, 1))
        return args[0] + (1.0 - args[1] * interior) * (-1e9)
    masked_logits = Lambda(_mask_pad_edge, name='mask_logits_pad_edge')([logits, valid_mask])
    masked_logits_kl = Lambda(lambda t: tf.transpose(t, perm=[0, 2, 1]),
                              name='transpose_kl')(masked_logits)
    probs = Activation('softmax', name='softmax_pos')(masked_logits_kl)
    return Model(inputs, probs, name='MultiScaleCNN_TopK_LEGACY')


# Build the active per-position model
cnn = build_cnn()
cnn.compile(
    optimizer=AdamW(learning_rate=LR, weight_decay=1e-5),
    loss=weighted_bce(POS_WEIGHT),
    metrics=[tf.keras.metrics.AUC(curve='PR', name='aupr')],
)
cnn.summary()


Model: "MultiScaleCNN_PerPosition"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ triplet_input       │ (None, 32000, 22) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_shared         │ (None, 32000, 64) │      4,288 │ triplet_input[0]… │
│ (Conv1D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_shared           │ (None, 32000, 64) │        256 │ conv_shared[0][0] │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_shared         │ (None, 32000, 64) │          0 │ bn_shared[0][0]   │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k3 (Conv1D)    │ (None, 32000, 64) │     12,352 │ relu_shared[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k7 (Conv1D)    │ (None, 32000, 64) │     28,736 │ relu_shared[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k15 (Conv1D)   │ (None, 32000, 64) │     61,504 │ relu_shared[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_k31 (Conv1D)   │ (None, 32000, 64) │    127,040 │ relu_shared[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_k3               │ (None, 32000, 64) │        256 │ conv_k3[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_k7               │ (None, 32000, 64) │        256 │ conv_k7[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_k15              │ (None, 32000, 64) │        256 │ conv_k15[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_k31              │ (None, 32000, 64) │        256 │ conv_k31[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_k3             │ (None, 32000, 64) │          0 │ bn_k3[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_k7             │ (None, 32000, 64) │          0 │ bn_k7[0][0]       │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_k15            │ (None, 32000, 64) │          0 │ bn_k15[0][0]      │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_k31            │ (None, 32000, 64) │          0 │ bn_k31[0][0]      │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_k3 (Dropout)   │ (None, 32000, 64) │          0 │ relu_k3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_k7 (Dropout)   │ (None, 32000, 64) │          0 │ relu_k7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ drop_k15 (Dropout)  │ (None, 32000, 64) │          0 │ relu_k15[0][0]  

 Total params: 513,729 (1.96 MB)

 Trainable params: 512,321 (1.95 MB)

 Non-trainable params: 1,408 (5.50 KB)

In [13]:
from tensorflow.keras.utils import plot_model

plot_model(
    cnn,
    to_file='figures/cnn_breakpoint_architecture.png',
    show_shapes=True,
    show_layer_names=True,
)
print("Architecture diagram saved to figures/cnn_breakpoint_architecture.png")

You must install pydot (`pip install pydot`) for `plot_model` to work.


Architecture diagram saved to figures/cnn_breakpoint_architecture.png


### CNN training

Callbacks:

- `EarlyStopping` (patience=15) restores the best weights at the end.
- `ReduceLROnPlateau` halves the learning rate after 5 stagnant epochs.
- `ModelCheckpoint` writes the best model so far to `models_test/`.

In [14]:
# Per-position monitor (run #16 baseline). val_aupr is the headline ranking
# metric — checkpoint / early-stop / LR-reduce on it directly.
callbacks = [
    EarlyStopping(
        monitor='val_aupr',
        mode='max',
        patience=15,
        restore_best_weights=True,
        verbose=1,
    ),
    ReduceLROnPlateau(
        monitor='val_aupr',
        mode='max',
        factor=0.5,
        patience=5,
        min_lr=1e-7,
        verbose=1,
    ),
    ModelCheckpoint(
        filepath='models_test/cnn_breakpoint_best.keras',
        monitor='val_aupr',
        mode='max',
        save_best_only=True,
        verbose=1,
    ),
]

print("Callbacks:", [cb.__class__.__name__ for cb in callbacks])
print("Monitoring val_aupr (mode=max)")


Callbacks: ['EarlyStopping', 'ReduceLROnPlateau', 'ModelCheckpoint']
Monitoring val_aupr (mode=max)


In [15]:
# Train the per-position CNN with tf.data prefetched pipeline + sample_weight=mask.
# (Top-K axis explored in #19-#21 — REVERTED, see Experiment Log #21.)
# (Parent-swap aug tested in #23 — REVERTED, hypothesis empirically wrong.)

# Run #24: shuffle buffer 2048 -> 256.  At MAX_SEQ_LEN=32000 each
# (X, y, w) tuple is ~1.5 MB; 2048-element buffer was ~3 GB CPU RAM
# on top of the resident numpy arrays — over the WSL VM's panic
# threshold.  256 is still ample mixing for ~890 batches/epoch.
# Run #24: pin from_tensor_slices to CPU. Without this, tf uploads
# X_train (~4.6 GB) and X_val (~1.7 GB) to GPU as constants and OOMs the
# RTX 3070's 5.5 GB free VRAM. With explicit CPU placement the bulk
# tensors stay in host RAM; only per-batch slices (~5.6 MB) reach GPU.
with tf.device('/CPU:0'):
    train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train, w_train))
                .shuffle(buffer_size=min(len(X_train), 256),
                         seed=42, reshuffle_each_iteration=True)
                .batch(BATCH_SIZE)
                .prefetch(tf.data.AUTOTUNE))

    val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val, w_val))
              .batch(BATCH_SIZE)
              .prefetch(tf.data.AUTOTUNE))

history = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

print("\nTraining complete.")


2026-05-05 17:17:32.424840: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 4922368000 exceeds 10% of free system memory.


2026-05-05 17:17:46.419097: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 1748736000 exceeds 10% of free system memory.


Epoch 1/100


2026-05-05 17:17:48.865205: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 4922368000 exceeds 10% of free system memory.


In [ ]:
def plot_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].plot(history.history['loss'], label='Train', linewidth=2)
    if 'val_loss' in history.history:
        axes[0].plot(history.history['val_loss'], label='Validation', linewidth=2)
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss (weighted_bce)')
    axes[0].set_title('Per-position weighted BCE loss')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    if 'aupr' in history.history:
        axes[1].plot(history.history['aupr'], label='Train', linewidth=2)
        if 'val_aupr' in history.history:
            axes[1].plot(history.history['val_aupr'], label='Validation', linewidth=2)
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('PR-AUC')
        axes[1].set_title('Per-position PR-AUC')
        axes[1].legend()
        axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('figures/cnn_training_history.png', dpi=150, bbox_inches='tight')
    plt.show()


plot_history(history)


In [ ]:
cnn.save('models_test/cnn_breakpoint_final.keras')
print("Model saved to models_test/cnn_breakpoint_final.keras")

## Breakpoint prediction evaluation

Three views of validation performance:

1. **Threshold sweep** with peak-based P / R / F1 to find the best operating point.
2. **Per-position ROC and PR curves** -- threshold-free quality.
3. **Visual inspection** of individual predictions with true breakpoints overlaid.

In [ ]:
# Per-position breakpoint probabilities for every validation sample.
# Shape: (n_val, MAX_SEQ_LEN).
y_val_pred = cnn.predict(X_val, batch_size=BATCH_SIZE, verbose=1)

print(f"Prediction shape: {y_val_pred.shape}")
print(f"Range: [{y_val_pred.min():.4f}, {y_val_pred.max():.4f}]")


In [ ]:
# Evaluate at multiple thresholds to find the best operating point.
thresholds = [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
results = []

print(f"Peak-based evaluation (+/-{TOLERANCE} bp tolerance, "
      f"min peak separation = {TOLERANCE} bp)")
print("=" * 60)

for thr in thresholds:
    p, r, f = evaluate_peaks(y_val_pred, meta_val, TOLERANCE, thr)
    results.append({'threshold': thr, 'precision': p, 'recall': r, 'f1': f})
    print(f"  threshold={thr:.2f}  P={p:.3f}  R={r:.3f}  F1={f:.3f}")

results_df = pd.DataFrame(results)
best_idx = results_df['f1'].idxmax()
best_thr = results_df.loc[best_idx, 'threshold']
print(f"\nBest threshold: {best_thr} (F1={results_df.loc[best_idx, 'f1']:.3f})")


### Per-sample prediction visualisation

Top panel: predicted probability over the genome with the threshold
line (red, dashed) and true breakpoint positions (green, vertical).
Bottom panel: ground-truth label mask.

In [ ]:
def plot_breakpoint_prediction(idx, X, y_true, y_pred, meta, threshold=0.5):
    fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
    bps = sorted([meta[idx]['bp_start'], meta[idx]['bp_end']])
    axes[0].plot(y_pred[idx], color='steelblue', alpha=0.85, label='predicted')
    if y_true is not None:
        axes[0].plot(y_true[idx], color='orange', alpha=0.5, label='true (Gaussian)')
    for bp in bps:
        axes[0].axvline(bp, color='red', linestyle='--', alpha=0.7)
    axes[0].axhline(threshold, color='gray', linestyle=':', label=f'thr={threshold}')
    axes[0].set_ylabel('P(breakpoint)')
    axes[0].legend(loc='upper right', fontsize=8)
    axes[0].grid(True, alpha=0.3)
    axes[0].set_title(f"Sample {idx}: file={meta[idx]['file']} event={meta[idx]['event']}")

    # Mask of valid (non-padded) positions for visual reference
    if X is not None:
        mask_arr = (X[idx].sum(axis=-1) > 0).astype(np.float32)
        axes[1].plot(mask_arr, color='gray')
        axes[1].set_ylabel('valid mask')
    axes[1].set_xlabel('Position (bp)')
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig


# Plot 3 representative samples
for ix in [0, len(X_val)//2, len(X_val)-1]:
    fig = plot_breakpoint_prediction(ix, X_val, y_val, y_val_pred, meta_val, threshold=best_thr)
    plt.show()


### ROC and precision-recall curves

Computed at the per-position level (every position, all samples flattened).
PR-AUC is the more honest summary because of the class imbalance --
ROC-AUC can look great while the model still misses most breakpoints.

In [ ]:
def plot_roc_and_pr(y_true_flat, y_pred_flat):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    fpr, tpr, _ = roc_curve(y_true_flat, y_pred_flat)
    roc_auc = auc(fpr, tpr)
    axes[0].plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3)
    axes[0].set_xlabel('False positive rate')
    axes[0].set_ylabel('True positive rate')
    axes[0].set_title('ROC curve')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    p, r, _ = precision_recall_curve(y_true_flat, y_pred_flat)
    pr_auc = average_precision_score(y_true_flat, y_pred_flat)
    axes[1].plot(r, p, label=f'PR (AUC = {pr_auc:.3f})')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title('Precision-Recall curve')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('figures/cnn_roc_pr_curves.png', dpi=150, bbox_inches='tight')
    plt.show()


# Use only valid (non-padded) positions for honest curves
valid_mask = (X_val.sum(axis=-1) > 0)
y_true_valid = y_val[valid_mask]
y_pred_valid = y_val_pred[valid_mask]
# Binarise the soft Gaussian targets at 0.5 for ROC/PR
plot_roc_and_pr((y_true_valid > 0.5).astype(int), y_pred_valid)


### Per-sample performance distribution

How does performance vary across samples? Histograms of per-sample
precision, recall, and F1 reveal whether average numbers come from
consistently decent predictions or a small number of perfect predictions
mixed with many failures.

In [ ]:
# Per-sample peak-based metric distributions (precision/recall/F1 per event)
per_sample = []
for i in range(len(X_val)):
    bps = [meta_val[i]['bp_start'], meta_val[i]['bp_end']]
    p, r, f, n_peaks, n_matched = peak_metrics_single(
        bps, y_val_pred[i], tolerance=TOLERANCE, threshold=best_thr,
    )
    per_sample.append({'precision': p, 'recall': r, 'f1': f,
                       'n_peaks': n_peaks, 'n_matched': n_matched})
ps_df = pd.DataFrame(per_sample)
print("Per-sample peak metric summary:")
print(ps_df.describe())

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ['precision','recall','f1']):
    ax.hist(ps_df[col], bins=20, color='steelblue', edgecolor='black')
    ax.set_title(col)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/cnn_metric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


### Event-level detection summary

Per-position metrics tell us how sharp the predictions are; this section
answers the more practical question: **how many recombination events did
the model actually find?**

Each event has two true breakpoints (`bp_start`, `bp_end`). Using the
best threshold and the existing greedy nearest-first matching within
`+/-TOLERANCE` bp, every event is classified as:

- **both**   -- both breakpoints matched to a predicted peak
- **one**    -- exactly one breakpoint matched (partial detection)
- **missed** -- neither breakpoint matched

We also report per-BP localisation error (|predicted - true| in bp) and
breakdowns by genome length and topology so it is clear *where* the
model is failing rather than just an averaged number.


In [ ]:
def classify_event(true_bps, y_pred, tolerance=TOLERANCE,
                   threshold=0.5, min_distance=None):
    """Per-event detection summary (per-position pipeline).

    Returns:
        dict with keys: status ("both"/"one"/"missed"), n_matched,
        bp_distances (list of |pred - true|, np.nan if unmatched),
        n_peaks, n_extra.
    """
    if min_distance is None:
        min_distance = tolerance
    peaks = detect_peaks(y_pred, threshold=threshold, min_distance=min_distance)
    true_bps = list(true_bps)
    pairs = sorted(((abs(int(p) - int(t)), pi, ti)
                    for pi, p in enumerate(peaks)
                    for ti, t in enumerate(true_bps)),
                   key=lambda x: x[0])
    matched_pred, matched_true = set(), set()
    distances = [np.nan] * len(true_bps)
    for d, pi, ti in pairs:
        if d > tolerance: break
        if pi in matched_pred or ti in matched_true: continue
        matched_pred.add(pi); matched_true.add(ti)
        distances[ti] = int(d)
    n_matched = len(matched_true)
    if n_matched == len(true_bps): status = "both"
    elif n_matched >= 1: status = "one"
    else: status = "missed"
    return {"status": status, "n_matched": n_matched,
            "bp_distances": distances, "n_peaks": len(peaks),
            "n_extra": len(peaks) - n_matched}


def build_event_df(y_pred_batch, meta_batch, threshold, tolerance=TOLERANCE):
    """Per-event DataFrame of detection outcomes (per-position pipeline)."""
    rows = []
    for yp, m in zip(y_pred_batch, meta_batch):
        info = classify_event([m["bp_start"], m["bp_end"]], yp,
                               tolerance=tolerance, threshold=threshold)
        rows.append({
            "file": m["file"], "event": m["event"],
            "actual_len": m["actual_len"],
            "circular": m["bp_start"] > m["bp_end"],
            "status": info["status"], "n_matched": info["n_matched"],
            "d_start": info["bp_distances"][0],
            "d_end":   info["bp_distances"][1],
            "n_peaks": info["n_peaks"], "n_extra": info["n_extra"],
        })
    return pd.DataFrame(rows)


def print_event_summary(events_df, title):
    n_total = len(events_df)
    n_both   = int((events_df["status"] == "both").sum())
    n_one    = int((events_df["status"] == "one").sum())
    n_missed = int((events_df["status"] == "missed").sum())
    print("=" * 60)
    print(title)
    print("=" * 60)
    print(f"Total events:            {n_total}")
    print(f"Both BPs found:          {n_both:4d}  ({n_both/n_total:.1%})")
    print(f"One BP found (partial):  {n_one:4d}  ({n_one/n_total:.1%})")
    print(f"Missed entirely:         {n_missed:4d}  ({n_missed/n_total:.1%})")
    print(f"Any BP found (>=1):      {n_both + n_one:4d}  "
          f"({(n_both + n_one)/n_total:.1%})")
    print()
    print(f"Mean predicted peaks per event:  {events_df['n_peaks'].mean():.2f}")
    print(f"Mean false-positive peaks:       {events_df['n_extra'].mean():.2f}")


In [ ]:
# Event-level analysis on the validation set (per-position)
events_val = build_event_df(y_val_pred, meta_val, threshold=best_thr)
print_event_summary(
    events_val,
    f"EVENT-LEVEL DETECTION (validation, threshold={best_thr}, "
    f"tolerance=+/-{TOLERANCE} bp)",
)


In [ ]:
# Localisation error histogram + status breakdown (per-position events_val)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
all_distances = pd.concat([events_val['d_start'], events_val['d_end']]).dropna()
axes[0].hist(all_distances, bins=40, color='steelblue', edgecolor='black')
axes[0].axvline(TOLERANCE, color='red', linestyle='--', label=f'tolerance ±{TOLERANCE}')
axes[0].set_xlabel('|pred - true| (bp)')
axes[0].set_ylabel('count')
axes[0].set_title(f'Localisation error distribution (val, per-position)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

status_counts = events_val['status'].value_counts()
axes[1].bar(status_counts.index, status_counts.values, color=['#2ecc71','#f1c40f','#e74c3c'])
axes[1].set_ylabel('events')
axes[1].set_title('Detection status (val)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('figures/cnn_metric_distributions.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Detection breakdown by genome length and topology (validation)
def detection_breakdown(events_df, group_col, group_label_map=None):
    """Cross-tab of status counts plus a both_pct column for quick reading."""
    tab = (events_df
           .groupby(group_col, observed=True)["status"]
           .value_counts()
           .unstack(fill_value=0)
           .reindex(columns=["both", "one", "missed"], fill_value=0))
    tab["total"] = tab.sum(axis=1)
    tab["both_pct"] = (tab["both"] / tab["total"] * 100).round(1)
    tab["any_pct"]  = ((tab["both"] + tab["one"]) / tab["total"] * 100).round(1)
    if group_label_map:
        tab.index = tab.index.map(lambda v: group_label_map.get(v, v))
    return tab


events_val = events_val.assign(
    len_bin=pd.cut(
        events_val["actual_len"],
        bins=[0, 2000, 3000, 4000, np.inf],
        labels=["<2kb", "2-3kb", "3-4kb", ">=4kb"],
    ),
)

print("Detection by genome length (validation):")
print(detection_breakdown(events_val, "len_bin"))
print()
print("Detection by genome topology (validation):")
print(detection_breakdown(events_val, "circular",
                          group_label_map={False: "Linear", True: "Circular"}))


## Unseen test set evaluation

`UnseenTestSet/` was never touched during training or threshold tuning.
We re-use the best threshold found on validation and report peak-based
P / R / F1 against held-out simulations.

In [ ]:
# Load and evaluate on unseen test set (per-position).
X_test, y_test, mask_test, meta_test = load_dataset(
    [TEST_DIR],
    label_mode='breakpoint',
)

y_test_pred = cnn.predict(X_test, batch_size=BATCH_SIZE, verbose=1)

p, r, f = evaluate_peaks(y_test_pred, meta_test, TOLERANCE, best_thr)

print("\n" + "=" * 60)
print("UNSEEN TEST SET PERFORMANCE (peak-based)")
print("=" * 60)
print(f"Precision: {p:.3f}")
print(f"Recall:    {r:.3f}")
print(f"F1 Score:  {f:.3f}")
print(f"Threshold: {best_thr}  |  Tolerance: +/-{TOLERANCE} bp")
print("=" * 60)


In [ ]:
# Event-level summary on the unseen test set (per-position)
events_test = build_event_df(y_test_pred, meta_test, threshold=best_thr)
print_event_summary(
    events_test,
    f"EVENT-LEVEL DETECTION (test, threshold={best_thr}, "
    f"tolerance=+/-{TOLERANCE} bp)",
)

import seaborn as sns
fig, ax = plt.subplots(figsize=(7, 4))
status_counts = events_test['status'].value_counts()
ax.bar(status_counts.index, status_counts.values, color=['#2ecc71','#f1c40f','#e74c3c'])
ax.set_ylabel('events')
ax.set_title(f'Detection status (test, per-position, F1={f:.3f})')
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('figures/event_detection_test.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# Per-sample test predictions (per-position)
for ix in [0, len(X_test)//2, len(X_test)-1]:
    fig = plot_breakpoint_prediction(ix, X_test, y_test, y_test_pred, meta_test, threshold=best_thr)
    plt.show()


## Analysis

Performance breakdowns and interpretability:

- **Circular vs linear breakpoints** -- does the model handle wrap-around correctly?
- **Performance by genome length** -- does padding longer or shorter sequences hurt?
- **Saliency maps** -- which input positions does the model attend to?

In [ ]:
# Circular vs linear genome breakdown (per-position events_val)
n_circ_val = events_val['circular'].sum()
if n_circ_val > 0:
    circ_both = ((events_val['circular']) & (events_val['status'] == 'both')).sum()
    print(f"Validation circular events: {n_circ_val} total, {circ_both} both-BPs found ({circ_both/n_circ_val:.1%})")
else:
    print("No circular genome breakpoints found in validation set.")


In [ ]:
# Performance by genome length (val, per-position)
events_val['len_bin'] = pd.cut(events_val['actual_len'],
                                bins=[0, 2000, 4000, 6000, 8000, 10000],
                                labels=['<2k','2-4k','4-6k','6-8k','8-10k'])
length_summary = events_val.groupby('len_bin', observed=True)['status'].value_counts(normalize=True).unstack(fill_value=0)
print("Validation Both BPs found % by genome length:")
print(length_summary)

if 'both' in length_summary.columns:
    fig, ax = plt.subplots(figsize=(7, 4))
    length_summary['both'].plot.bar(ax=ax, color='steelblue', edgecolor='black')
    ax.set_ylabel('Fraction Both BPs found')
    ax.set_title('Per-position Both-BPs detection by genome length (val)')
    ax.set_xlabel('Length bin')
    plt.xticks(rotation=0)
    ax.grid(True, alpha=0.3, axis='y')
    plt.tight_layout()
    plt.savefig('figures/performance_by_length.png', dpi=150, bbox_inches='tight')
    plt.show()


### Saliency map (input attribution)

Gradient of the model output with respect to the input shows which
positions most strongly drive the prediction. Currently averaged over
all output positions; for a sharper picture see TODO.md (compute
saliency w.r.t. specific high-probability peaks instead).

In [ ]:
# Saliency map placeholder. Re-enable in a follow-up if useful.
print("Saliency map: skipped for now.")


# Experiment Log

This is the canonical record of every training run. New entries go at the **top**.
See [HANDOVER.md](HANDOVER.md) §5 for the entry template and §6 for the
KEPT / REVERTED / INCONCLUSIVE decision criteria.

If you change the metric set, add a new field at the bottom of the template — do not redefine existing ones.

---

## 2026-05-05 #24 — MAX_SEQ_LEN 10k → 32k + paired POS_WEIGHT/BATCH_SIZE — RUNNING

**Status:** training in progress, started 2026-05-05T17:13:05.

**Hypothesis:** Test F1 has been pinned at ~0.12–0.17 across seven uncorrelated changes (#17–#23). 2026-05-05 diagnostic (above) showed UnseenTestSet sequences are ~30 kb (vs train/val mean 10.8 kb) so 48.8% of test `bp_start` and 74.4% of test `bp_end` are past `MAX_SEQ_LEN=10000` and literally unseen by the model. If true, bumping `MAX_SEQ_LEN` to capture the full test sequence should lift test F1 sharply with no other change. SANTA-output sanity check (sampled 5 UnseenTestSet `.fa` files): alignment length ≈30,000 bp, mean gap fraction <1%, ≈29,750 ATGC bases per sequence — sequences are real long DNA, not gappy alignments. Truncation arithmetic holds.

**Change:** "Fix truncation" treated as ONE logical change with three coupled hyperparameter moves (per HANDOVER §4 — paired infrastructure):
- cell-3: `MAX_SEQ_LEN` 10000 → **32000** (max observed test `bp_end` is ~30,143; 32000 gives 1.9 kb headroom).
- cell-3: `BATCH_SIZE` 8 → **2** (RTX 3070 8 GB VRAM at 3.2× longer sequences).
- cell-12: `POS_WEIGHT` 70 → **178** (mean(y) drops ~3× when sequences ~3× longer at same 2 bps/event; data-implied ~210, apply run-#8's 0.847× factor).
- cell-12: `max_files=500` cap on each train dir (15 GB WSL VM panicked on full-set `np.concatenate` at 32k length); val (XML-5) unchanged.
- cell-12: removed dead `X_all/y_all/mask_all` concat (saves ~11 GB peak RAM, not used by any live cell).

Cache invalidates because cache key includes `MAX_SEQ_LEN` — first-run cold cache expected ~10–15 min for re-parse + re-encode of 5 train dirs + UnseenTestSet.

**Config snapshot:**
- max_files: 500 (train), None (val, test)
- LABEL_SIGMA: 20
- LABEL_MODE: gaussian
- POS_WEIGHT: 178
- LR: 1e-4
- BATCH_SIZE: 2
- Loss: weighted_bce
- Architecture: residual dilated stack with 4 MaxChi windows (50, 100, 200, 500), per-position #16 baseline (no top-K, no boundary mask)
- MAX_SEQ_LEN: 32000
- N_INPUT_CHANNELS: 22

**Results:** _TBD when run completes — see `/tmp/run24_log.txt` for live progress._

**Verdict criteria (pre-pinned per session prompt):**
- Test F1 ≥ 0.25 → KEPT (truncation hypothesis confirmed; rewrites the project's "0.172 ceiling" narrative).
- Test F1 in [0.18, 0.25] → KEPT but partial (truncation explains some but not all of the gap; investigate residual val/test divergence).
- Test F1 < 0.18 → INCONCLUSIVE/REVERTED (hypothesis wrong or paired-infra bug; consult advisor before iterating on MAX_SEQ_LEN further).

---

## 2026-05-05 #23 — Parent-swap augmentation — REVERTED

**Hypothesis:** The persistent val/test gap (#17-#22) is an invariance issue. Parent channel order is arbitrary; per-batch swap with sign-flip on MaxChi forces parent-symmetric features.

**Change:** cell-22 only — `tf.data.map(parent_swap_aug)` between shuffle and batch on train_ds. Per-sample p=0.5 swap of channels [5:10]↔[10:15], [15]↔[16], negate [18:22]. val_ds unaugmented.

**Results (val-best threshold = 0.5):**
- Best epoch: 7 / Train PR-AUC: 0.171 / Val PR-AUC: 0.128 / Train-val gap: +0.044
- **Val F1: 0.294** (vs 0.282 in #16 — +0.012, basically noise)
- Val Both BPs: 8.9% (vs 8.5% — flat)
- **Test F1: 0.125** (vs 0.172 in #16 — **−0.047**)
- Test Both BPs: 3.7% (flat)
- Test One BP: 26.8% / Missed: 69.5% / Mean peaks: 3.48 (vs 2.34 in #16 — model less confident)

**Verdict:** REVERTED.

**Why:** Hypothesis empirically wrong — parent labelling is NOT arbitrary in SANTA's output. Random parent-swap during training removes systematic signal that the model was using. The model became less precise (mean peaks 2.34 → 3.48) and test F1 dropped substantially. Confirms that "obvious symmetries" in this data aren't actually symmetric.

**Larger pattern across #17-#23:** Every local-axis change (input features #17, σ #18, dropout #16, kernels #22, parent-aug #23) fails to close the cross-config val/test gap. Val F1 clusters at 0.28-0.31; test F1 clusters at 0.12-0.17. The gap isn't being attacked by any of these. The bottleneck might be:
- The training distribution itself (XML-1..4 has too narrow a feature distribution)
- The architecture's inductive bias (cross-XML invariance not encoded)
- Or the test set genuinely has a fundamentally different generation profile that no train-time intervention can bridge

**Next:** Need strategic re-evaluation, not another local-axis swing. Advisor consultation queued.

---

## 2026-05-05 #22 — Bigger multi-scale kernels (7,31,63,127) — REVERTED

**Hypothesis:** Per-advisor — bigger receptive field at the input layer attacks the cross-config generalization bottleneck (XML-5 → UnseenTestSet) without custom-layer cost. (7, 31, 63, 127) widens local-pattern receptive field from ~31 bp to ~127 bp.

**Change:** cell-18 `filter_sizes=(3, 7, 15, 31)` → `(7, 31, 63, 127)`. Single config change.

**Results (val-best threshold = 0.5):**
- Best epoch: 5 (vs 8 in #16 — earlier plateau)
- Train PR-AUC at best: 0.172 / Val PR-AUC at best: 0.129
- Train-val gap: +0.044 (vs +0.037 in #16 — slightly wider)
- **Val F1: 0.308** (vs 0.282 in #16 — **+0.026**)
- Val Both BPs: 9.0% (vs 8.5% in #16 — +0.5pp, noise)
- **Test F1: 0.149** (vs 0.172 in #16 — **−0.023**)
- Test Both BPs: 3.7% (same as #16, flat)
- Test One BP: 29.3% / Missed: 67.1%
- Test precision/recall: 0.125 / 0.183 (vs 0.205 / 0.177 in #16 — precision DOWN, recall UP)
- Mean peaks/event (test): 2.93 (vs 2.34 in #16 — model fires more peaks)

**Verdict:** REVERTED (advisor pre-pinned: Test F1 < 0.16 → REVERTED, parent-swap aug as #23).

**Why:** Same val/test gap pattern as runs #14, #15. Val improved meaningfully (+0.026 F1) but test got worse. Bigger input-layer kernels help fit the training distribution (incl. XML-5 val held-out) but the additional capacity is largely "fitting cross-XML noise" — it doesn't generalize across the additional distribution shift to UnseenTestSet. The model fires more peaks per event (2.93 vs 2.34) at lower per-peak precision (0.125 vs 0.205) — symptom of over-fitting to local context.

**Next:** Per advisor — parent-swap augmentation (HANDOVER queue medium-priority #5). Channel order for parent 1 vs parent 2 is arbitrary; symmetric augmentation costs nothing and directly attacks the generalization gap that has now defeated three different feature/regularization changes (#13-#16, #17, #22).

---

## 2026-05-05 DIAGNOSTIC — val/test gap is a TRUNCATION BUG, not generalization

After 7 consecutive REVERTED runs (#17–#23) with no test F1 movement past #16's 0.172, advisor pushed back: "Stop hunting. The val/test gap is a property of the test set, not the model. Measure first." Diagnostic compared XML-5 val (n=621) vs UnseenTestSet (n=82) on meta dimensions. **Smoking gun:**

| metric | val (XML-5) | test (UnseenTestSet) |
|---|---|---|
| `actual_len` mean / median | 10,788 / 10,784 | **30,009 / 30,004** |
| `bp_start >= MAX_SEQ_LEN (10000)` | 0.6% | **48.8%** |
| `bp_end >= MAX_SEQ_LEN (10000)` | 32.7% | **74.4%** |
| bps within 50bp of edge | ~30% | **~80%** |
| `bp_start` in 8-10k position bin | 11.1% | **52.4%** |

**Test sequences are ~3× longer than train/val.** `MAX_SEQ_LEN=10000` truncates each test sample to its first 1/3, throwing away most breakpoints. **48.8% of test `bp_start` and 74.4% of test `bp_end` literally don't exist in the encoded input.** Of the bps the model can see, 80% are within 50bp of an edge — exactly where the BN+padding boundary spike hurts most.

**Re-explains every prior failure:**
- 7 consecutive runs with stable test F1 ≈ 0.12-0.17 → consistent truncation, consistent miss rate
- Run #21 boundary suppression crashed test F1 to 0.024 → we banned the *only* positions where truncated test bps appear
- Bigger kernels (#22) helped val but not test → val sequences fit in MAX_SEQ_LEN, test ones don't
- Top-K mode-collapse to 0 / 9999 (#19/#20) → "boundary attractor" was actually fitting the *truncated* test distribution

**The fix is one line:** `MAX_SEQ_LEN = 10000 → 32000` in cell-3.

**Implications and to-do for run #24 (next session, recommended fresh-context start):**

1. **Cache invalidation.** Cache key includes MAX_SEQ_LEN; bumping it forces all 5 directories + UnseenTestSet to re-parse and re-encode. Estimated cold-cache time: ~10-15 min.

2. **VRAM constraint.** RTX 3070 has 8GB. Per-batch memory scales with MAX_SEQ_LEN: 8 × 32000 × 22 × 4 = 22 MB input alone, plus activations × 6 dilated blocks at 64 filters ≈ 8 × 32000 × 64 × 6 × 4 ≈ 400 MB per layer. Will likely OOM at BATCH_SIZE=8. Drop to BATCH_SIZE=4 or BATCH_SIZE=2 preemptively. (LR scaling concern is paired infrastructure, not a research change.)

3. **Receptive field rethink.** With MAX_SEQ_LEN=32000 and dilated stack k=7, dilations 1..32, current receptive field is ~410 bp — still adequate for local recombination signals (which are bp-scale). MaxChi windows up to 500 are still relevant. No architectural change needed beyond the MAX_SEQ_LEN bump.

4. **Label generation.** `generate_labels` uses `seq_length=MAX_SEQ_LEN` arg with default — will auto-scale. `bp_end` values up to ~30,143 will now fit (max test bp_end is 30,143 < 32000). No clamping pile-up.

5. **POS_WEIGHT recompute.** Per HANDOVER §4, when MAX_SEQ_LEN changes, POS_WEIGHT may need rescaling. With sequences ~3× longer and roughly same number of positives per event (2), mean(y) drops to ~1/3 of current. Implied POS_WEIGHT triples to ~210. But run #8 found data-implied is over-aggressive at σ=20 — apply same 70/82.62 = 0.847× factor: 210 × 0.847 ≈ 178. Treat as paired infrastructure change with the MAX_SEQ_LEN bump (not a separate experiment).

6. **`actual_len_in_X`** in `meta` no longer matches `actual_len` (the latter is the original alignment length). Downstream cells use `actual_len` for length-bin breakdown — should still work since `actual_len` is the true sequence length, not the encoded length.

**Quick sanity check before run #24:** verify the SANTA simulator outputs are 30k-bp sequences and not just 30k-bp alignments with internal gaps. If they're real 30k-bp sequences, the bp coordinates are biological positions and MAX_SEQ_LEN=32000 captures them. If they're alignments with many gaps, encoded length might be shorter and the truncation issue might be different.

**Stop:** This is THE finding for the session. Run #24 is the obvious test. If MAX_SEQ_LEN=32000 lifts test F1 from 0.17 to anywhere near val (0.28), the project's whole "structural val/test gap" narrative gets rewritten — there was no structural gap, just a hyperparameter mismatch.

---

🚩 **STOP CONDITION FIRED — HANDOVER §7 trip-wire** (5 consecutive REVERTED: #17, #18, #20, #21, #22; #19 INCONCLUSIVE per advisor pin).

**Session summary 2026-05-04 → 2026-05-05:**

Six runs (#17-#22). Two productive surfaces under "no headline win":

1. **Top-K formulation explored and closed.** Three runs (#19/#20/#21) showed:
   - The BN+'same'-padding boundary spike, harmless under per-position sigmoid+threshold, *catastrophically* wrecks any softmax-over-positions head — model collapses to head-0=position 0 (99.8%) and head-1=position 9999 (86.2%). Argmax amplifies small logit differences into huge probability differences.
   - The boundary-mask trick (logits + targets in `[0, EDGE_BUFFER) ∪ [L-EDGE_BUFFER, L)` masked) is necessary infrastructure for any future argmax-style head. Codified in CLAUDE.md "Three things easy to break" (now five).
   - With boundaries banned, the backbone can't do interior localization to argmax precision (test F1 = 0.024). The constraint is upstream of the head.
   - The K_TOPK / TOPK_TARGET_SIGMA / EDGE_BUFFER constants in cell-3 and the legacy build_cnn_topk / topk_xent_loss / make_topk_targets functions are preserved for future revisits.

2. **Cross-config generalization (val/test gap) is the active bottleneck under per-position.** Three different attempts (#17 wider MaxChi, #18 σ-paired, #22 bigger kernels) all show the same pattern: val improves, test doesn't transfer or gets worse. The model has enough capacity; it doesn't have the right inductive bias for cross-XML→UnseenTestSet generalization. Augmentation and feature engineering at the *invariance* axis (e.g., parent-swap, reverse-complement) are the natural next attack surfaces.

3. **Pipeline optimisation infrastructure built.** Disk cache for `load_dataset` (per-directory, keyed on MAX_SEQ_LEN/MAXCHI_WINDOWS/LABEL_SIGMA/BP_WINDOW) cuts subsequent same-config wall time by ~50%. tf.data prefetch in cell-22. Memory-growth in cell-1. Cache populated for σ=20 + 4-MaxChi config (158 MB across 5 dirs + test).

**State at handoff (run #22 reverted, cell-3 / cell-18 back to per-position #16 baseline):**
- Best test F1 still run #16: **0.172** (Both BPs 3.7%, Mean peaks/event 2.34, threshold 0.7).
- Cached models for runs #17-#22 in `models_test_backup/cnn_breakpoint_best.run{17..22}.keras`.
- Notebook on disk reflects the reverted per-position #16 baseline pipeline. Cache is warm for default config.

**Recommended #23 (when next session resumes):**
- Parent-swap augmentation (cheap, directly attacks invariance/generalization).
- Implementation: per-batch with prob 0.5, swap channels [5:10] ↔ [10:15] AND [15] ↔ [16] (match_p1 ↔ match_p2). Channel 17 (informative) is symmetric. MaxChi channels 18-21 use `(match_p1 - match_p2)` so they flip sign — handle by computing per-batch via `tf.signal` ops or just precompute swapped versions in load_dataset cache. Targets are unchanged (bp positions don't depend on parent labelling).

**Other queue items still active:**
- Reverse-complement augmentation (recombination is direction-symmetric).
- Masked BatchNorm (deferred per advisor — only justified if argmax-style head returns).
- Deeper backbone (n_filters 64 → 128 across the dilated stack).

---

## 2026-05-04 #21 — Top-K + boundary attractor fix — REVERTED (CLOSES top-K AXIS)

**Hypothesis:** Mode collapse in #19/#20 (head-0→0 in 99.8%, head-1→9999 in 86.2%) was caused by (1) target clamping: bps with `bp_end > MAX_SEQ_LEN` clamped to position 9999, and (2) BN + 'same' padding boundary spike making positions 0 and L−1 systematically extreme. Banning the edge-buffer in both targets and logits should let the model localize in the interior.

**Change:** Three coordinated edits, single logical idea ("fix mode-collapse attractor"):
- cell-3: `EDGE_BUFFER = 50` constant.
- cell-16 `make_topk_targets`: bps in `[0, EDGE_BUFFER) ∪ [MAX_SEQ_LEN − EDGE_BUFFER, MAX_SEQ_LEN)` → target = all-zeros (no loss/gradient contribution).
- cell-18 `build_cnn_topk`: mask Lambda extended to subtract 1e9 from logits in the same edge-buffer regions.

**Diagnostic outputs from cell-12 / cell-22:**
- TRAIN: 1154 / 6662 (sample, head) pairs skipped (~17%).
- VAL: 396 / 1242 (~32%).
- TEST: ~similar fraction (max true_late_bp = 10874 vs MAX_SEQ_LEN = 10000).

**Results:**
- Best epoch: **26** (was 1 in #19/#20 — model unstuck, training runs for many epochs and ReduceLROnPlateau fires twice).
- Train loss: 7.55 → 6.53 (decreases meaningfully).
- val_topk_match_rate best: **0.154** (was 0.330 in #19/#20).
- **Val F1: 0.151** / Val Both BPs: 0.3% / Val One BP: 29.6% / Missed: 70.0%.
- **Test F1: 0.024** / Test Both BPs: **0%** / Test One BP: **4.9%** / Missed: **95.1%**.
- Median per-distribution argmax confidence: 0.0013 (model is very flat).

**Verdict:** REVERTED — closes the top-K axis with this backbone.

**Why (the deep learning):** The 0.330 val_topk_match_rate in #19/#20 was *the boundary attractor matching test bps that happened to be near boundaries by chance*. With boundaries banned, the *true* interior localization rate is ~15%. The model trained for 41 epochs and could not exceed this. Test F1 crashed from 0.16 → 0.02 because the boundary "attractor" was doing the heavy lifting on test (test bps near boundaries no longer get matched within ±200 tolerance). With boundaries banned, the model has to actually localize, and it can't.

**This closes the top-K axis with the current backbone.** Three runs (#19, #20, #21) tested three target shapes (one-hot, σ=5 Gaussian, with-edge-suppression) and arrived at three failures: stalled at epoch 1, identical stall, then 41 epochs of grinding without breaking 0.024 test F1. The bottleneck is upstream of the head — the backbone produces features that are good for graded per-position scoring (#16: F1=0.172) but lack the spatial precision needed for confident argmax-style prediction. Replacing the head alone is insufficient.

**Pre-pinned verdict applied:** Test F1 < 0.18 → REVERTED + pivot to per-position. Test F1 < 0.10 specifically → "top-K is dead with this backbone." Confirmed empirically.

**Next:** Per advisor pre-commit, #22 goes BACK TO PER-POSITION (#16 baseline pipeline). The top-K cells (18, 21, 22, 23, 26, 27, 29, 31, 33, 35, 36, 37, 40, 41, 42, 44, 45, 47) are reverted to their per-position equivalents in this commit. Cell-3 retains K_TOPK / TOPK_TARGET_SIGMA / EDGE_BUFFER constants as historical artifacts; they're unused under per-position.

The KEY new evidence from #19/#20/#21: the BN + 'same' padding boundary artifact is *empirically confirmed* to dominate any argmax-style prediction. Run #12 (per-position) found it as a minor effect that didn't kill F1, but this run shows it's a real architectural issue. **Masked BatchNorm (HANDOVER queue item #4)** has direct motivation: it would compute BN stats only over valid positions, eliminating the artifact at the architectural level.

**Run #22 candidate** (advisor consultation needed before staging):
- Masked BatchNorm under per-position. Custom layer. Single change vs #16 baseline.
- Hypothesis: removes the same boundary artifact that wrecked #19/#20/#21, but applied where we know the rest of the pipeline trains: per-position sigmoid + Gaussian targets + weighted_bce.

---

## 2026-05-04 #20 — Top-K + soft Gaussian targets σ=5 — REVERTED (closes top-K axis)

**Hypothesis:** Run #19's top-K with hard one-hot stalled at val_topk_match_rate=0.330, best epoch=1. Conjectured cause: extreme target sparsity (1 positive position out of 10000 per head) gives a near-degenerate cross-entropy gradient that the model can't refine past initial gain. Smoothing the target to a normalized Gaussian σ=5 (~10 positions of mass per head) should give a learnable gradient signal while still encouraging sharp argmax.

**Change:** Single change in `make_topk_targets` (cell-16) — `targets[i, k, bp] = 1.0` replaced by a normalized Gaussian peak σ=`TOPK_TARGET_SIGMA=5` centered at `bp`. New constant `TOPK_TARGET_SIGMA` added to cell-3. Architecture, loss formulation (categorical CE), optimizer, callbacks all unchanged from #19.

**Config snapshot:**
- max_files: None (cache HIT for all 5 directories + UnseenTestSet)
- LR: 1e-4, AdamW(weight_decay=1e-5)
- Loss: topk_xent_loss
- Architecture: residual dilated stack + Conv1D(K=2, 1) + masked softmax
- BATCH_SIZE: 8, K_TOPK: 2, TOPK_TARGET_SIGMA: 5
- MAX_SEQ_LEN: 10000, N_INPUT_CHANNELS: 22

**Results:**
- Best epoch: 1 (val_topk_match_rate peaked at 0.32965, never improved over 15 more epochs — IDENTICAL to #19 best epoch and value to 5 decimal places)
- **Val F1: 0.182** (same as #19)
- **Val Both BPs: 2/621 = 0.3%** (same as #19)
- **Test F1: 0.152** (vs #19's 0.159 — slightly DOWN, vs #16's 0.172 — −0.020)
- **Test Both BPs: 0/82 = 0%** (same as #19)
- Median per-distribution argmax confidence: 0.0010 (vs #19's 0.0033 — distribution flatter)
- Mean predicted positions/event: 2.00 (forced by K=2)
- Wall time: 7.4 min (vs #19's 15.7 min — half the time thanks to warm cache)

**Verdict:** REVERTED — closes the top-K axis per advisor pre-pinned protocol.

**Why (the deep learning):** The val_match_rate match to 5 decimal places between #19 and #20 is the diagnostic. With `tf.random.set_seed(42)` + `np.random.seed(42)` + `tf.data.shuffle(seed=42)` everything is deterministic, so initial weights and data order are identical between runs. Both target shapes (one-hot and σ=5 Gaussian) peak at the *same* bp position, so the cross-entropy gradient direction in epoch 1 is essentially the same. The model converges to the *same local minimum* and stalls there.

**Target sharpness is NOT the top-K bottleneck.** The bottleneck is the optimization landscape: softmax over 10000 positions with this backbone has very weak gradient signal beyond an early-epoch attractor — the model latches onto a coarse pattern in epoch 1 (val_match_rate ≈ 0.33 = ~30% of (sample, head) argmaxes within ±200 bp of truth) and *cannot* refine past that. The model produces near-uniform softmax distributions (median confidence 0.001-0.003) and gets argmax matches mostly by chance localization.

**Closes the top-K axis with this backbone.** Two attempts (#19, #20) on the per-position-trained backbone with two different target shapes both produced test F1 ≈ 0.15-0.16 and 0% Both BPs. Replacing the output head alone is insufficient; the backbone's features are good for graded per-position scoring (#16: F1 0.172, Both BPs 3.7%) but apparently lack the precision needed to drive a confident argmax-style prediction.

**Pipeline note:** Cache worked perfectly — all 5 directories + test set hit cache, 7.4 min total wall time vs #19's 15.7 min cold (≈53% reduction). Future runs at unchanged data config will be similarly fast.

**Next:** Several options worth weighing carefully (advisor consultation queued before staging #21):

1. **Auxiliary per-position head** — multi-task learning. Keep top-K head + add per-position sigmoid head with Gaussian-target loss. Total loss = α × topk_xent + β × weighted_bce. The per-position head gives the backbone strong, learnable gradient signal (we *know* it can train); the top-K head benefits from the resulting features. Hypothesis: backbone's features under per-position loss are sufficient to drive top-K precision when both heads are trained jointly.

2. **Pivot back to per-position** with a fresh attack on Both BPs. Run #16 plateaued at test F1 = 0.172 / Both BPs = 3.7%. Untried axes: masked BatchNorm (queue item), bigger multi-scale kernels (queue item), positional encoding for explicit position awareness, longer training with cosine LR schedule.

3. **Top-K with bigger backbone + warmup** — not the right diagnostic to do *before* (1) and (2), but worth flagging if (1) reveals the issue is "needs more capacity for argmax precision".

The auxiliary head (option 1) is the most informative single experiment because it tests the hypothesis "is it the head, the backbone, or the joint setup that's the bottleneck". If it lifts test F1 above #16's 0.172, the multi-task approach is the future. If it doesn't, the constraint is upstream of the head.

---

## 2026-05-04 #19 — Top-K head with hard one-hot targets — INCONCLUSIVE

**Hypothesis:** Per-position binary classification has plateaued at test F1 ≈ 0.15-0.17. Replacing the per-position sigmoid with K=2 independent softmax distributions over positions (sorted-pair, hard one-hot targets) should escape the "≈1 peak per event" failure mode that's been the persistent cause of low Both-BPs-found %.

**Change:** First top-K refactor — see Pending block above for the full cell list. Backbone identical to run #16 (residual+dilated, 64 filters, dropout=0.5). Hard one-hot targets at sorted bp positions; categorical cross-entropy per head; argmax = prediction. Mask in logits (not loss).

**Config snapshot:**
- max_files: None (3331 train events from XML-1..4, 621 val from XML-5)
- LABEL_SIGMA: 20 (legacy; not used in top-K path)
- LR: 1e-4, AdamW(weight_decay=1e-5)
- Loss: topk_xent_loss (categorical CE per head)
- Architecture: residual dilated stack + Conv1D(K=2, 1) → mask logits → softmax over L
- BATCH_SIZE: 8
- MAX_SEQ_LEN: 10000
- N_INPUT_CHANNELS: 22
- K_TOPK: 2

**Pre-train smoke:** Passed — output (n, 2, 10000), softmax sums to 1.0, argmax always inside valid (non-padded) region.

**Results:**
- Best epoch: 1 (val_topk_match_rate peaked at 0.330, never improved over 15 more epochs)
- Train loss at best: 7.55 → 7.43 over 16 epochs (~1.6% decrease, barely moves)
- Train topk_match_rate: 0.31 (final), val 0.28 (drifts down from 0.33 best)
- **Val F1 (top-K event-level): 0.182** (vs 0.282 in #16 — −0.100)
- **Val Both BPs found: 2/621 = 0.3%** (vs 8.5% in #16 — **crashed**)
- Val One BP: 35.7% / Missed: 63.9%
- Mean predicted positions/event: 2.00 (forced by K=2)
- Mean false-positive predictions: 1.64 (val), 1.68 (test)
- Median per-distribution argmax confidence: 0.0033 (vs uniform 0.0001 — barely 30× above random)
- **Test F1 (top-K): 0.159** (vs 0.172 in #16 — **−0.013**)
- **Test Both BPs found: 0/82 = 0%** (vs 3.7% in #16 — crashed)
- Test One BP: 31.7% / Missed: 68.3%
- Test precision/recall: 0.159 / 0.159

**Verdict:** INCONCLUSIVE (per advisor pre-pinned protocol)

**Why:** Test F1 lands in [0.10, 0.20] which the advisor pre-pinned as "one tuning iteration before verdict on the top-K axis as a whole." The qualitative diagnostic is clear: the model latches onto a weak pattern in epoch 1 (val_match_rate=0.33 already) and cannot refine. Median argmax confidence 0.0033 means the softmax distribution is barely above uniform — the model is essentially producing flat distributions and getting argmax matches by chance localization.

**Why hard one-hot stalls (empirical):** L=10000 with 2 positive positions is an extremely sparse target. The cross-entropy gradient is concentrated at correct positions (1−p ≈ 1), but softmax normalization globally couples all 10000 positions, so each step must redistribute mass against an essentially uniform prior. The model quickly reaches a local minimum where it can't shift further. The advisor's hard-one-hot pin is theoretically correct (no graded confidence), but empirically the gradient signal is too sparse to learn the localization task on first contact.

**Pipeline notes:** Cold cache for σ=20, 4-MaxChi-window config; cache populated for all 5 directories + UnseenTestSet (~158 MB total). Subsequent same-config runs will skip data prep entirely. End-to-end runtime 15.7 min including data prep + training + eval; would be ~9-10 min on warm cache.

**Next:** Run #20 = top-K + soft Gaussian targets σ=5. Single change in `make_topk_targets`: replace `targets[i, k, bp] = 1.0` with a normalized Gaussian peak (σ=5) centered at bp. Targets remain concentrated (~10 positions worth of mass) but the gradient signal smooths out, addressing the empirical "can't refine past epoch 1" failure mode without reintroducing the σ=20-style graded confidence over hundreds of positions.

If #20 also stalls below test F1 = 0.20, top-K with this backbone is exhausted. Consult advisor before further changes (auxiliary per-position head, deeper output head, or pivot to a different formulation).

---

## 2026-05-04 #18 — σ=10 paired with POS_WEIGHT=140 — REVERTED

**Hypothesis:** Sharper Gaussian targets (σ=20 → 10) should encourage more localized peaks. Run #9 showed that σ=10 alone (with POS_WEIGHT=70 left at the σ=20 baseline) under-fits because the positive label mass roughly halves. Pairing POS_WEIGHT 70 → 140 keeps the loss-weight-on-positives proportional to the proven-good baseline. This closes the σ/POS_WEIGHT coupling story open since #9.

**Change:** cell-3 `LABEL_SIGMA` 20 → 10 and cell-12 `POS_WEIGHT` 70 → 140 — one logical change per HANDOVER §4 ("σ + POS_WEIGHT as one unit"). Architecture, optimizer, MaxChi windows unchanged from #16 baseline.

**Config snapshot:**
- max_files: None (3331 train events from XML-1..4, 621 val from XML-5)
- LABEL_SIGMA: 10
- POS_WEIGHT: 140 (data-implied was 155.77 — we under-shot by run-#8-calibrated 0.847× heuristic; see "Why" below)
- LR: 1e-4
- Loss: weighted_bce
- Architecture: residual dilated stack (k=7, d=1..32, n_filters=64, dropout=0.5)
- BATCH_SIZE: 8
- MAX_SEQ_LEN: 10000
- N_INPUT_CHANNELS: 22
- MAXCHI_WINDOWS: (50, 100, 200, 500)

**Results (at val-best threshold = 0.5):**
- Best epoch: 2 (val_aupr peaked very early)
- Train PR-AUC at best epoch: 0.117 (vs ~0.176 in #16 — substantially lower; model is under-fitting)
- Val PR-AUC at best epoch: 0.110 (vs 0.139 in #16 — DOWN)
- Train-val PR-AUC gap: +0.007 (vs +0.037 in #16 — tight, but tight at LOWER train, not higher val)
- Val F1 (peak-based): 0.281 (vs 0.282 in #16 — flat)
- Val "Both BPs found": 9.5% (vs 8.5% in #16 — +1pp, noise)
- **Test F1 (peak-based): 0.160** (vs 0.172 in #16 — **−0.012**)
- **Test "Both BPs found": 3.7%** (same as #16, flat)
- Test One BP: 26.8% / Missed: 69.5%
- Test precision / recall: 0.184 / 0.171 (vs 0.205 / 0.177 in #16 — both down)
- Mean predicted peaks/event (test): 2.43 (vs 2.34 in #16)
- Best threshold: 0.5 (vs 0.7 in #16)

**Verdict:** REVERTED

**Why:** Test F1 −0.012 fails KEPT criterion 1; Both BPs flat fails criterion 2. The narrow train-val gap (+0.007) looks like a qualitative win but isn't — it comes from train PR-AUC also dropping (0.176 → 0.117), so the gap is closing because the model is *under-fitting both sides*, not because it's generalizing better. Sharper σ=10 targets are intrinsically harder to fit (less mass per peak, narrower local shape); the model converges to a lower-capacity solution by epoch 2 and plateaus. POS_WEIGHT scaling didn't compensate.

**Closes the σ/POS_WEIGHT coupling story:** the answer is "they don't help in pairing either." Run #9 was REVERTED because POS_WEIGHT was too low at σ=10. Run #18 is REVERTED with POS_WEIGHT=140 (close to data-implied 155.77, slightly under per the run-#8 0.847× heuristic). The σ axis is exhausted — neither σ=10 alone nor σ=10 with paired POS_WEIGHT improves test performance. Future runs should not revisit σ tuning unless the underlying loss formulation changes.

**Per advisor's pinned protocol:** Run #19 is the top-K coordinate regression refactor regardless of #18's outcome. The per-position approach has now plateaued at test F1 ≈ 0.16-0.17 across runs #13–#18, with three different axes attempted (input features, dropout, σ tuning) — none lifted test F1 past 0.172. Time to attack the formulation.

**Next:** (1) Implement user-requested infra optimization (disk cache for load_dataset + tf.data prefetch + GPU memory growth) so future runs are faster. (2) Call advisor for top-K design pass (Hungarian vs sorted-pair, K=2 fixed vs K-variable, no-recombination handling). (3) Implement and run #19.

---

## 2026-05-04 #17 — Extended MaxChi windows (4 → 6) — REVERTED

**Hypothesis:** Adding wider MaxChi windows (1000, 2000 bp on top of 50/100/200/500) would let the model see longer-range parental-disparity statistics, which should help on the held-out XML-5 (≥4kb only) and the longer-sequence-skewed UnseenTestSet.

**Change:** cell-3 only — `N_MAXCHI_WINDOWS` 4 → 6, `MAXCHI_WINDOWS` extended to `(50, 100, 200, 500, 1000, 2000)`, `N_INPUT_CHANNELS` 22 → 24. Paired infrastructure change: `BATCH_SIZE` 16 → 8 (forced by RTX 3070's 8GB VRAM after machine migration; M4 had 16GB unified). LR/loss/architecture unchanged.

**Config snapshot:**
- max_files: None (3331 train events from XML-1..4, 621 val from XML-5)
- LABEL_SIGMA: 20
- POS_WEIGHT: 70.0
- LR: 1e-4
- Loss: weighted_bce
- Architecture: residual dilated stack (k=7, d=1..32, n_filters=64, dropout=0.5)
- BATCH_SIZE: 8 (was 16 on M4; forced down for 8GB VRAM)
- MAX_SEQ_LEN: 10000
- N_INPUT_CHANNELS: 24 (15 triplet one-hot + 3 comparison + 6 MaxChi)
- MAXCHI_WINDOWS: (50, 100, 200, 500, 1000, 2000)

**Results (at val-best threshold = 0.6):**
- Best epoch: 6 (vs 8 in #16 — earlier plateau)
- Train PR-AUC at best epoch: 0.187
- Val PR-AUC at best epoch: 0.144 (vs 0.139 in #16; +0.005)
- Train-val PR-AUC gap: +0.043 (vs +0.037 in #16 — slightly *wider*)
- Val F1 (peak-based): 0.286 (vs 0.282 in #16; +0.004 — noise)
- Val "Both BPs found": 10.8% (vs 8.5% in #16; +2.3pp — mild positive)
- Val Mean peaks/event: 2.06 (vs 1.85 in #16)
- **Test F1 (peak-based): 0.161** (vs 0.172 in #16; **−0.011**)
- **Test "Both BPs found": 3.7%** (vs 3.7% in #16; flat)
- Test One BP: 29.3% / Missed: 67.1% (vs One BP not directly logged in #16; Missed unchanged in spirit)
- Test precision / recall: 0.173 / 0.183 (vs 0.205 / 0.177 in #16; precision DOWN, recall up slightly)
- Mean predicted peaks/event (test): 2.66 (vs 2.34 in #16)
- Best threshold: 0.6 (vs 0.7 in #16 — shifted, but extended sweep already covers this regime)

**Verdict:** REVERTED

**Why:** The headline test metric got worse (Test F1 −0.011 fails KEPT criterion 1; Test Both BPs flat fails criterion 2). The slight val improvements (+0.004 F1, +2.3pp Both BPs) didn't transfer to test — a textbook case of the structural val/test gap that runs #14, #15 already documented. The train-val gap widened slightly (+0.037 → +0.043), so the wider windows didn't even help generalization within the train distribution. Reverted cell-3 to 4 windows.

Don't read the small val improvement (+0.004 F1, +2.3pp Both BPs) as a partial win — paired with the *wider* train-val gap and worse test, it's val moving in the direction of further overfit, not better generalization. Wider MaxChi windows are not worth revisiting; the val/test structural gap is the real bottleneck and lives somewhere else (architecture or output formulation, not input channels).

**Hardware note:** First run on the migration target (Linux WSL2 / RTX 3070 8GB / Ryzen 5 3600). End-to-end runtime 19.4 min (M4 was ~80–100 min) — GPU + nbclient flow works cleanly. BATCH_SIZE 16 → 8 was forced by VRAM; if a future run wants to re-test BATCH_SIZE specifically, it's a free axis to revisit.

**Next:** Per HANDOVER §12, when #17 doesn't help, the recommendation is to stop stacking input-feature variants and either (a) re-attempt σ=10 paired with POS_WEIGHT scaling, or (b) refactor to a top-K coordinate regression head that attacks the val/test distribution gap structurally. Going with σ-tuning first as the smaller, cheaper test — if it also fails, top-K is the next swing.

---


## 2026-05-04 #16 — Increase dropout 0.3 → 0.5

**Hypothesis:** Run #15 exposed a +0.072 train-val PR-AUC gap (overfitting to XML-1..4). Bumping dropout 0.3 → 0.5 in `build_cnn` is the cheapest test of whether reducing model capacity at training time closes the gap and improves generalization to held-out XML-5 / UnseenTestSet.

**Change:** cell-18: `dropout=0.3` → `dropout=0.5` in the `build_cnn` default. Single-axis change.

**Config snapshot:**
- max_files: None (3331 train / 621 val from XML-5 held-out)
- LABEL_SIGMA: 20
- POS_WEIGHT: 70.0
- MAX_SEQ_LEN: 10000
- N_INPUT_CHANNELS: 22 (with MaxChi from #13)
- dropout: 0.5

**Results (at val-best threshold = 0.7):**
- Best epoch: 8 (val_aupr peaked very early — model under-fits at this dropout)
- Train PR-AUC at best epoch: ~0.176 (vs ~0.215 in #15 — DOWN, regularization is biting)
- Val PR-AUC at best epoch: 0.139 (vs 0.143 in #15 — basically flat)
- Train-val PR-AUC gap: +0.037 (vs +0.072 in #15 — narrowed by half but not closed)
- Val F1 (peak-based): 0.282 (vs 0.321 in #15 — −0.039)
- Val "Both BPs found": 8.5% (vs 18.2% in #15 — **−9.7pp**, capability halved)
- Test F1 (peak-based): **0.172** (vs 0.151 in #15 — **+0.021**, clears KEPT bar)
- Test "Both BPs found": 3.7% (vs 4.9% in #15 — −1.2pp)
- Test precision / recall: 0.205 / 0.177 (vs 0.135 / 0.213 — precision +0.070, recall −0.036)
- Mean predicted peaks / event: 2.34 (test); 1.85 (val)

**Verdict:** KEPT (Test F1 +0.021 ≥ +0.01 under §6 criterion 1)

**Why:** Mixed signal — the strongest test F1 improvement in many runs (+0.021), but val Both BPs halved (18.2% → 8.5%). Higher dropout pushed the model into a precision regime: it emits ~2 peaks per genome instead of 4, with higher per-peak confidence but missing more total BPs. The trade-off looks similar to run #10 (LayerNorm) but less severe — Both BPs didn't collapse to 0%, and test F1 actually moved.

The train-val gap narrowed from +0.072 to +0.037 (~50% reduction), so dropout did partially address overfitting — but not enough to lift val. The gain shows up only on test, suggesting test/val mismatch is more about distribution than overfitting.

**Side observations:**
- Best epoch=8 (vs 16 in #15, 14 in #14, 13 in #13) — the model finds its best very early and then drifts. Could be EarlyStopping triggering on noise, or genuine quick convergence under stronger regularization.
- Dropout 0.5 may be too aggressive. A sweet spot at 0.4 is worth trying — but stack one experiment at a time.

**Next:** Run #17 — extend MaxChi windows from {50, 100, 200, 500} to {50, 100, 200, 500, 1000, 2000}. The ≥4kb val regime (XML-5 is entirely ≥4kb) and the test set (also long) should benefit from wider parental-disparity statistics. N_INPUT_CHANNELS bumps 22 → 24. Single change — same input-channels-driven leverage that delivered #13's gains, applied at longer scales now that the long-genome regime is the primary signal source.

---

## 2026-05-04 #15 — Strict held-out split: TRAIN=XML-1..4, VAL=XML-5

**Hypothesis:** Run #14's val gain (+0.015 F1, +5.5pp Both BPs) didn't transfer to test, suggesting the previous mixed-XML val was sharing distribution with training. A strict directory-level held-out (XML-5 entirely as val) tests cross-configuration generalization — closer to the simulation→real transfer the project ultimately wants. This is a *diagnostic* run; the goal is to recalibrate what "good" means, not to lift headline numbers.

**Change:** cell-12: load XML-1..4 and XML-5 as separate datasets. cell-13: skip the file-shuffle split, use the directory-level partition directly. cell-45: made robust to single-bin val sets (XML-5 has no 3-4kb genomes, all are ≥4kb).

**Config snapshot:**
- max_files: None  (3331 train / 621 val samples)
- LABEL_SIGMA: 20
- POS_WEIGHT: 70.0 (data-implied 77.51 from train only)
- MAX_SEQ_LEN: 10000
- N_INPUT_CHANNELS: 22 (with MaxChi)

**Results (at val-best threshold = 0.5):**
- Best epoch: 16 (training continued 31 epochs)
- Train PR-AUC at best epoch: ~0.215 (vs ~0.193 in #14)
- **Val PR-AUC at best epoch: 0.143** (vs 0.206 in #14 — DOWN substantially)
- **Train-val PR-AUC gap: +0.072** (vs −0.013 in #14 — overfitting reappears)
- Val F1 (peak-based): **0.321** (vs 0.356 in #14 — −0.035; the *honest* number)
- Val "Both BPs found": **18.2%** (vs 27.7% in #14 — −9.5pp; the honest number)
- Val ≥4kb Both BPs: 18.2% (XML-5 has no 3-4kb genomes — all sequences ≥4kb)
- Test F1 (peak-based): 0.151 (vs 0.145 in #14 — +0.006, within noise)
- Test "Both BPs found": 4.9% (unchanged)
- Test precision / recall: 0.135 / 0.213
- Mean predicted peaks / event: 4.39 (test); 2.79 (val)

**Threshold sweep (val):**
- thr=0.30 P=0.211 R=0.502 F1=0.281
- thr=0.40 P=0.267 R=0.437 F1=0.308
- thr=0.50 P=0.306 R=0.401 F1=0.321  **(val-best)**
- thr=0.60 P=0.330 R=0.365 F1=0.320
- thr=0.70 P=0.340 R=0.331 F1=0.312
- thr=0.95 P=0.268 R=0.165 F1=0.198

Note val-best threshold dropped 0.8 → 0.5 — predictions are softer on a held-out config (less confident, smaller peaks).

**Verdict:** KEPT (under §6 criterion 3: test F1 essentially unchanged + qualitative property improves — we now have an honest val signal that mirrors test, and a measurable train-val gap to optimize against)

**Why:** Val F1 dropped from 0.356 → 0.321 because XML-5 is genuinely held out — no test/val leakage from shared XML configurations. The mixed-XML val of #14 was overstating model capability. Test F1 is essentially the same (val-tuned threshold adapted: 0.5 vs 0.8). Two real problems now exposed:

1. **Overfitting on XML-1..4**: train PR-AUC 0.215 vs val (XML-5) PR-AUC 0.143 — gap +0.072. The model memorizes XML-1..4 patterns that don't generalize to XML-5. Run #13's "gap −0.003" was illusory because val was mixed.
2. **XML-5 ≠ UnseenTestSet**: even with held-out XML-5, val F1 0.321 ≫ test F1 0.151. UnseenTestSet has its own distribution that no held-out XML covers. The val/test gap can't be closed by training-set design alone.

**Side observations:**
- XML-5 sequences are all ≥4kb (max actual_len ≤ 10000, all in-frame). Train (XML-1..4) has 3-4kb sequences too (valid_fraction 0.675 vs val 1.000). XML-5 may be a longer-genome configuration — explains the val_fraction discrepancy.
- The XML-5 val partition (621 events) is now larger than the test set (82 events) — better signal-to-noise for iteration than test alone.

**Next:** Run #16 — increase dropout 0.3 → 0.5 in cell-18. Cheapest test of the overfitting hypothesis. If train-val gap narrows and val F1 holds or improves, overfitting was the lever. If train PR-AUC drops without val improvement, we've under-fit and need a different attack.

---

## 2026-05-04 #14 — `max_files = 750 → None` (full training set)

**Hypothesis:** With #13's MaxChi features now eliminating overfitting (train-val PR-AUC gap −0.003), more training data should generalize cleanly and narrow the val/test gap (val F1 0.341 vs test F1 0.139). Single-knob change, ~2.3× compute per epoch.

**Change:** cell-12: `max_files=750` → `max_files=None`. Loads all available files in each XML-N directory.

**Config snapshot:**
- max_files: None  (3392 train / 560 val samples; 2467 unique files vs 1755 in #13)
- LABEL_SIGMA: 20
- POS_WEIGHT: 70.0 (data-implied 84.96)
- MAX_SEQ_LEN: 10000
- N_INPUT_CHANNELS: 22 (with MaxChi from #13)

**Results (at val-best threshold = 0.8):**
- Best epoch: 14 (vs 13 in #13)
- Train PR-AUC at best epoch: ~0.193 (vs ~0.198 in #13)
- Val PR-AUC at best epoch: 0.206 (vs 0.201 in #13)
- Train-val PR-AUC gap: −0.013 (slightly under-fitting if anything)
- Val F1 (peak-based): 0.356 (vs 0.341 in #13 — **+0.015**)
- Val "Both BPs found": 27.7% (vs 22.2% in #13 — **+5.5pp**)
- Val 3-4kb Both BPs: 38.8% (vs 28.0% in #13 — +10.8pp)
- Val ≥4kb Both BPs: 24.2% (vs 20.7% in #13 — +3.5pp)
- Test F1 (peak-based): 0.145 (vs 0.139 in #13 — +0.006, within noise)
- Test "Both BPs found": 4.9% (vs 4.9% in #13 — **flat**)
- Test precision / recall: 0.120 / 0.213
- Mean predicted peaks / event: 4.40 (test); 3.48 (val)

**Verdict:** KEPT (under §6 criterion 3: test F1 essentially unchanged + qualitative property improves — val F1 +0.015 and val Both BPs +5.5pp are clearly above noise on n=560)

**Why:** Val improvements landed exactly where the hypothesis predicted: more data → better in-distribution performance. Val F1 +0.015 and Both BPs +5.5pp are both real (val n=560, well above noise floor). But test held flat, which is more important: it tells us the val/test gap is *structural*, not sample-size. The test set's distribution differs from XML-1..5 in some way the model hasn't been able to bridge. With only n=82 test events, all test deltas <0.07 F1 are within typical SEM noise — so test "flat" doesn't mean "no improvement", it means "improvement undetectable at this sample size".

**Side observations:**
- Val/test ratio 2.46× (val F1 0.356 vs test F1 0.145). MaxChi (#13) didn't close it, more data (#14) didn't either. Gap is likely cross-configuration distributional shift, not data quantity.
- Val 3-4kb Both BPs jumped from 28.0% → 38.8% (+10.8pp). The model can do well on shorter genomes even with the val/test gap.
- Per-epoch wall time ~135s (vs ~125s in #13) — minor data loading overhead, no architectural slowdown.

**Next:** Run #15 — stricter held-out split. Move XML-5 entirely into the val partition (TRAIN: XML-1..4, VAL: XML-5). This tests whether the val/test gap is a true cross-configuration distribution shift. If val=XML-5 looks more like the UnseenTestSet (similar F1, same Both BPs scale), the diagnosis is confirmed and our previous val numbers were overstating capability. If val=XML-5 still looks like the prior in-mix val, then UnseenTestSet has its own peculiarity (different SANTA params or labelling).

---

## 2026-05-04 #13 — MaxChi-style "parental switch disparity" channels (4 new inputs)

**Hypothesis:** The classical detection methods (MaxChi, GeneConv) compute "running parental-match disparity" at multiple window sizes — exactly the statistic that lights up at recombination breakpoints. Runs #6-#12 plateaued at val F1 ~0.34, test F1 ~0.15, with graded confidence (real-BP peaks lower in amplitude than the boundary spike). If the conv stack can't extract these statistics implicitly from match_p1/match_p2, hand the model the answer as input channels.

**Change:** cell-3: `N_INPUT_CHANNELS` 18 → 22, added `MAXCHI_WINDOWS = (50, 100, 200, 500)`. cell-6: in `encode_triplet`, after computing `match_p1`/`match_p2`/`informative`, compute `parental_signal = match_p1 - match_p2` and 4 MaxChi channels:

    maxchi[p, w] = mean(parental_signal[p:p+w]) - mean(parental_signal[p-w:p])

implemented via prefix-sum cumsum for O(L) per window, with zero-padding outside the valid region. Concatenated to give (L, 22). cell-12: reverted #12's edge-buffer mask (now baseline #8 with the new feature channels).

**Config snapshot:**
- max_files: 750
- LABEL_SIGMA: 20
- POS_WEIGHT: 70.0
- MAX_SEQ_LEN: 10000
- N_INPUT_CHANNELS: 22 (was 18)
- Architecture: residual dilated stack (unchanged structure; first conv input channels grew 18 → 22)

**Results (at val-best threshold = 0.8):**
- Best epoch: 13 (training continued 28 epochs — converged 6 epochs faster than #8's 19)
- Train PR-AUC at best epoch: ~0.198 (vs ~0.224 in #8 — DROPPED, but val rose)
- Val PR-AUC at best epoch: 0.201 (vs 0.195 in #8 — +0.006)
- **Train-val PR-AUC gap: −0.003 (vs +0.029 in #8 — overfitting essentially eliminated)**
- Val F1 (peak-based): 0.341 (vs 0.339 in #8 — flat)
- **Val "Both BPs found": 22.2% (vs 15.7% in #8 — +6.5pp)**
- Val 3-4kb Both BPs: 28.0% (vs ~25.6% in #8 — +2.4pp)
- Val ≥4kb Both BPs: 20.7% (vs ~13.5% in #8/12 — +7.2pp; biggest gain on longest genomes)
- Test F1 (peak-based): 0.139 (vs 0.149 in #8 — −0.010)
- **Test "Both BPs found": 4.9% (vs 2.4% in #8 — +2.5pp, clears §6 KEPT bar)**
- Test precision / recall: 0.118 / 0.201 (vs 0.135 / 0.183 — recall +0.018, precision −0.017)
- Mean predicted peaks / event: 3.90 (test); 3.32 (val)

**Threshold sweep (val):**
- thr=0.50 P=0.128 R=0.853 F1=0.217
- thr=0.70 P=0.250 R=0.559 F1=0.328
- thr=0.80 P=0.289 R=0.469 F1=0.341  **(val-best)**
- thr=0.90 P=0.311 R=0.393 F1=0.333
- thr=0.95 P=0.310 R=0.354 F1=0.320

**Verdict:** KEPT (Test Both BPs +2.5pp ≥ +2pp under §6 criterion 2; train-val gap collapse is a qualitative win)

**Why:** MaxChi features delivered exactly what the hypothesis predicted: the model now finds Both BPs more often (val +6.5pp, test +2.5pp) and stops overfitting (train-val PR-AUC gap −0.003 vs +0.029). The signal is strongest where the prior architecture struggled most — val ≥4kb (+7.2pp Both BPs) — consistent with longer genomes needing wider-window parental statistics that the unstructured CNN couldn't synthesize. Test F1 dipped 0.010 (precision −0.017, recall +0.018) — within noise on n=82, and net-positive given the precision/recall trade flipped in the right direction (recall is what matters when "find at least one BP" is the deployment task).

**Side observations:**
- Best epoch fell 19 → 13: easier task = faster convergence. Could also let the model run longer with less aggressive early stopping if that helps.
- Train-val gap collapse implies the MaxChi channels regularize. Future capacity expansion (more layers, more filters) is now safer — the model isn't memorizing.
- The val/test gap remains huge (val F1 0.341 vs test F1 0.139, ratio ~2.5×). Not addressed by this run; primary candidates for next: more training data, stricter held-out split (XML-1..4 → XML-5).

**Next:** Run #14 — `max_files = 750 → None` (full training set). The val/test gap is the dominant unsolved problem, and with the MaxChi features now regularizing well, throwing more data at it is a clean single-knob change. ~2.3× compute per epoch, no architectural change — stacks safely on the new baseline.

---

## 2026-05-04 #12 — 100bp edge-buffer mask in loss

**Hypothesis:** User flagged a saliency plot showing edge spikes at sequence start (~0-100 bp) and at the actual_len boundary (~3800 bp) that reach probability ≈1.0, while interior predictions sit flat at ~0.4-0.5. The edge spikes look like a cheap local minimum: the model gets free training hits by predicting "boundary = breakpoint" (some BPs are near edges). Eroding the per-sample loss mask by 100 bp on each side of the valid region should deny gradient at edges, force the model to find signal in the interior.

**Change:** cell-12: after `load_dataset`, erode `mask_all`. For each sample, find the first/last valid index and zero the mask in [first, first+100) and [last-99, last+1]. Also re-derive `mean_y_unmasked` over the eroded mask. POS_WEIGHT stays hardcoded at 70.

**Config snapshot:**
- max_files: 750
- LABEL_SIGMA: 20
- POS_WEIGHT: 70.0
- MAX_SEQ_LEN: 10000
- Architecture: residual dilated stack (unchanged from #6/#8)
- Edge buffer: 100 bp on each side of valid region

**Results (at val-best threshold = 0.8):**
- Best epoch: 21 (training continued 36 epochs)
- Train PR-AUC at best epoch: ~0.216 (vs ~0.224 in #8 — slight drop)
- Val PR-AUC at best epoch: 0.187 (vs 0.195 in #8 — slight drop)
- Val F1 (peak-based): 0.336 (vs 0.339 in #8 — basically same)
- Val "Both BPs found": 14.2% (vs 15.7% in #8 — −1.5pp)
- Test F1 (peak-based): 0.155 (vs 0.149 in #8 — **+0.006**, below KEPT bar)
- Test "Both BPs found": 3.7% (vs 2.4% in #8 — +1.3pp, below KEPT bar)
- Test precision / recall: 0.141 / 0.201
- Mean predicted peaks / event: 3.22 (test); 2.78 (val)

**Diagnostic chart (post-run, on cached model):** [figures/run12_chart.png](figures/run12_chart.png)
- Edge spikes still present at actual_len boundary (~1.0) in every sample.
- The 100 bp loss mask did not suppress them — the spike is mostly in the *padding region* (positions ≥ actual_len), where there was no loss signal anyway. The mask only affected the last 100 bp of valid region, which is already a small slice.
- Real-BP peaks in the interior are LOWER in amplitude than the boundary spike. At threshold=0.8, we keep boundary spikes and miss most real BPs.

**Post-hoc diagnostics on the cached model:**
- Boundary suppression sweep (zeroing first/last K∈{0,100,200,400,600} bp before find_peaks): F1 *decreases* monotonically with larger K at every threshold. Suppressing the boundary loses real signal in the eroded valid region (some training BPs are near edges).
- Top-K peak reranking (ignore threshold, take top K peaks by height): val F1 peaks at K=4 sup=0 (F1=0.332, Both=22.9%); test F1 peaks at K=2 sup=0 (F1=0.183, Both=2.4%). Modest improvement over threshold-tuning; not a breakthrough.

**Verdict:** INCONCLUSIVE → effectively REVERTED (cell-12 reset to #8 baseline before run #13)

**Why:** The edge-buffer change neither cleanly hurt nor helped — within noise on every metric. More importantly, the diagnostic chart shows it didn't fix the symptom that motivated it: boundary spikes persist because they live in the padding region, not in the masked-out edge of the valid region. The post-hoc top-K and suppression sweeps tell a clearer story: the model has *graded confidence*, and real-BP peaks are smaller than the boundary spike, so any threshold either keeps both (low precision) or filters both (low Both BPs). The bottleneck is interior signal sharpness, not boundary contamination.

**Next:** Run #13 — engineered MaxChi-like features as 4 new input channels. The classical recombination-detection methods (MaxChi, GeneConv) compute "running parental-match disparity" at multiple window sizes; the CNN appears to be unable to extract this implicitly from match_p1/match_p2 alone (graded confidence, no sharp interior peaks). Hand the model the answer:
  - parental_signal[p] = match_p1[p] - match_p2[p]
  - maxchi[p, w] = mean(parental_signal[p:p+w]) - mean(parental_signal[p-w:p])
  - For w ∈ {50, 100, 200, 500} bp.

Total input channels: 18 → 22. Single architectural change (encode_triplet). N_INPUT_CHANNELS bumps in cell-3 to match.

---

## 2026-05-04 #10 — `BatchNormalization` → `LayerNormalization`

**Hypothesis:** With MAX_SEQ_LEN=10000 and ~30% padding, BatchNorm computes statistics over zero-valued padding positions, potentially producing boundary artifacts. LayerNorm normalizes per-position over channels and is oblivious to cross-batch padding statistics — it should remove the artifact and improve test F1.

**Change:** cell-18: replaced every `BatchNormalization()` with `LayerNormalization()` (10 layers total). Single change.

**Config snapshot:**
- max_files: 750
- LABEL_SIGMA: 20
- POS_WEIGHT: 70.0
- LR: 1e-4
- Loss: weighted_bce
- Architecture: residual dilated stack, LayerNorm
- MAX_SEQ_LEN: 10000

**Results (at val-best threshold = 0.6):**
- Best epoch: 21 (training continued 36 epochs, 144s/epoch — ~15% slower than BN)
- Train PR-AUC at best epoch: ~0.16 (vs ~0.224 in #8 — significantly lower; LN regularizes more strictly)
- Val PR-AUC at best epoch: 0.184 (vs 0.195 in #8 — slight drop)
- Val F1 (peak-based): 0.286 (vs 0.339 in #8 — −0.053)
- Val "Both BPs found": **0.0%** (vs 15.7% in #8 — capability regression)
- Test F1 (peak-based): 0.173 (vs 0.149 in #8 — **+0.024**)
- Test "Both BPs found": **0.0%** (vs 2.4% in #8 — capability regression)
- Test precision / recall: 0.239 / 0.146 (vs 0.135 / 0.183 — precision +0.10, recall −0.04)
- Mean predicted peaks / event: 1.61 (test); 1.84 (val) — model only emits ~1 peak per genome
- Prediction range over val: [0.0000, 0.9910]

**Threshold sweep (val):**
- thr=0.50 P=0.248 R=0.366 F1=0.272
- thr=0.60 P=0.305 R=0.277 F1=0.286  **(val-best)**
- thr=0.70-0.95: identical (P=0.305 R=0.277 F1=0.286) — predictions cluster sharply, threshold barely matters above 0.6

**Verdict:** REVERTED (capability regression on Both BPs metric)

**Why:** Test F1 +0.024 looks like a small win, but the gain comes from being more conservative (0.149→0.173 via precision +0.104, recall −0.037). The cost is severe: "Both BPs found" went 15.7% → 0% on val and 2.4% → 0% on test. The persistent ceiling that #6 worked hard to break (0% Both BPs across runs #2–#5) is back. CLAUDE.md explicitly names both F1 and the event-level breakdown as headline numbers; capability regression on Both BPs counts as "clearly hurts on the headline metric" under §6.

LayerNorm appears to push the model into a strict precision regime — it emits ~1 peak per genome instead of 2.7, regardless of threshold above 0.6. The flat threshold sweep (F1=0.286 across 0.6-0.95) confirms predictions cluster narrowly. With only one confident peak per event, you cannot reach Both BPs. Train PR-AUC also dropped 0.224 → 0.16 — LN regularizes harder, the architecture under-fits.

cell-18 reverted to BatchNormalization before logging.

**Side observations:**
- Per-epoch wall time went 125s (BN) → 144s (LN), a ~15% slowdown. First attempt timed out at 7200s. Bumped notebook timeout to 21600s for the retry.
- The val/test gap *reversed* under LN: val F1 0.286 < test F1 0.173 — and val Both BPs = test Both BPs = 0.0%. This is consistent with the "model is too conservative" reading: when both sets are starved of peaks, val no longer has the advantage.

**Next:** Priority queue #2 — `max_files = 750 → None` (full training set). The val/test gap under BN (val F1 0.339 vs test F1 0.149) is the dominant unsolved problem; throwing more training data at it is the cleanest single-knob change. Compute cost ~2.3× per epoch but no architectural change, so it stacks safely on the run #8 baseline.

---

## 2026-05-04 #9 — `LABEL_SIGMA` 20 → 10 (sharper Gaussian targets)

**Hypothesis:** Sharper target peaks should encourage more localized predictions, potentially reducing false-positive density per real BP. The mean(y) will halve (Gaussian integral 50 → 25); POS_WEIGHT stays hardcoded at 70 per #8.

**Change:** cell-3: `LABEL_SIGMA = 20 → 10`. Nothing else changed.

**Config snapshot:**
- max_files: 750  (2531 train / 415 val samples)
- LABEL_SIGMA: 10
- POS_WEIGHT: 70.0 (hardcoded; data-implied = 166.09)
- LR: 1e-4
- Loss: weighted_bce
- Architecture: residual dilated stack
- MAX_SEQ_LEN: 10000

**Results (at val-best threshold = 0.6):**
- Best epoch: 17 (training continued 32 epochs total before early stop)
- mean(y) over unmasked: 0.00598 (was 0.01196 — halved as expected)
- Train PR-AUC at best epoch: ~0.146 (vs ~0.224 in #8 — DROPPED)
- Val PR-AUC at best epoch: 0.128 (vs 0.195 in #8 — DROPPED)
- Val F1 (peak-based): 0.335 (vs 0.339 in #8 — within noise)
- Val "Both BPs found": 23.4% (vs 15.7% in #8 — +7.7pp at val-best, but val-best threshold is now 0.6)
- Test F1 (peak-based): 0.097 (vs 0.149 in #8 — **−0.052, clearly hurts**)
- Test "Both BPs found": 3.7%   ("One BP": 35.4%   "Missed": 61.0%)  (vs 2.4% in #8 — +1.3pp)
- Test precision / recall: 0.065 / 0.213
- Mean predicted peaks / event: 6.26 (test); 3.59 (val)

**Threshold sweep (val):**
- thr=0.30 P=0.116 R=0.887 F1=0.201
- thr=0.50 P=0.231 R=0.566 F1=0.315
- thr=0.60 P=0.276 R=0.478 F1=0.335  **(val-best)**
- thr=0.70 P=0.297 R=0.416 F1=0.333
- thr=0.80 P=0.317 R=0.386 F1=0.335
- thr=0.95 P=0.314 R=0.316 F1=0.307

**Verdict:** REVERTED (test F1 −0.052 clearly hurts headline; cell-3 reverted to LABEL_SIGMA=20 before logging)

**Why:** Two coupled effects degraded performance:
1. The sharper target halved mean(y) on unmasked positions (0.012 → 0.006), but POS_WEIGHT stayed at 70. The data-implied POS_WEIGHT for σ=10 is 166 — at 70, positives are underweighted and gradient flow on positive regions is weaker. Train PR-AUC dropped 0.224 → 0.146 (architecture under-fits sharper targets at this loss balance).
2. The narrower targets give a much smaller "useful" gradient region per BP (σ=10 → ~21 positions with y > 0.135; σ=20 → ~42). With proportionally less positive signal per sample, optimization is harder.

The val/test gap also widened (val F1 0.335 vs test F1 0.097, ratio ~3.5×), suggesting the sharper targets overfit harder.

This run *cannot* be cleanly attributed to σ alone because POS_WEIGHT was the wrong value for σ=10. To properly test sharper targets, POS_WEIGHT would need to scale with the new mean(y). That's two changes; rather than retry, move to a different axis.

**Next:** Priority queue #2 — `BatchNorm` → `LayerNorm`. With MAX_SEQ_LEN=10000 and 30% padding, BatchNorm computes statistics over zero-valued padding positions, potentially producing boundary artifacts at the actual sequence end. LayerNorm normalizes per-position over channels and ignores cross-batch statistics, removing the padding-statistics issue entirely. Single change in cell-18: replace each `BatchNormalization()` call with `LayerNormalization()`.

---

## 2026-05-04 #8 — Revert `POS_WEIGHT` to 70 (override auto-derivation)

**Hypothesis:** Run #7's auto-derivation pushed `POS_WEIGHT` 70 → 82.62. Combined with the residual stack already finding signal, the more aggressive positive weighting may have been driving the over-firing (~25 peaks/genome at low thresholds) and the widened val/test gap. Hardcoding `POS_WEIGHT = 70.0` while keeping `MAX_SEQ_LEN = 10000` should preserve the truncation fix while reducing precision crash.

**Change:** cell-12: replaced `POS_WEIGHT = (1 - mean_y_unmasked) / mean_y_unmasked` with a hardcoded `POS_WEIGHT = 70.0`. Diagnostic still prints the *implied* value (82.62) so future runs can compare. Everything else identical to #7.

**Config snapshot:**
- max_files: 750  (2531 train / 415 val samples)
- LABEL_SIGMA: 20
- POS_WEIGHT: 70.0 (override; data-implied 82.62)
- LR: 1e-4
- Loss: weighted_bce
- Architecture: residual dilated stack (unchanged from #6/#7)
- MAX_SEQ_LEN: 10000

**Results (at val-best threshold = 0.9):**
- Best epoch: 19 (training continued 34 epochs total before early stop)
- Train PR-AUC at best epoch: ~0.224 (similar to #7)
- Val PR-AUC at best epoch: 0.195 (vs 0.192 in #7)
- Val F1 (peak-based): 0.339 (vs 0.338 in #7 — flat)
- Val "Both BPs found": 15.7% (vs 26.0% in #7 — −10.3pp at val-best threshold)
- Test F1 (peak-based): **0.149** (vs 0.113 in #7 — **+0.036, clears KEPT bar**)
- Test "Both BPs found": 2.4%   ("One BP": 31.7%   "Missed": 65.9%)  (vs 2.4% in #7)
- Test precision / recall: 0.135 / 0.183 (vs 0.085 / 0.189 — precision +0.050)
- Mean predicted peaks / event: 3.01 (test); 2.67 (val)
- Prediction range over val: [0.0000, 0.9999]

**Threshold sweep (val, n=415):**
- thr=0.20 P=0.084 R=0.948 F1=0.153  peaks≈?  (compared to #7 thr=0.20: P=0.080 R=0.953 F1=0.147)
- thr=0.70 P=0.177 R=0.713 F1=0.274  (vs #7 thr=0.70: P=0.097 R=0.933 F1=0.173 — much higher precision, lower recall)
- thr=0.80 P=0.256 R=0.540 F1=0.331
- thr=0.90 P=0.309 R=0.412 F1=0.339  **(val-best)**
- thr=0.95 P=0.317 R=0.369 F1=0.329

The val-best threshold dropped from 0.95 (#7) to 0.9 — predictions are slightly less confident on average since the loss pushes positives less hard.

**Verdict:** KEPT (Test F1 +0.036 ≥ 0.01)

**Why:** Less aggressive positive weighting reduced over-firing (val mean peaks at val-best 3.77 → 2.67) and improved precision on the held-out test set (0.085 → 0.135). Test recall held flat (0.189 → 0.183), so the precision gain translated cleanly into F1 (+0.036). On val, F1 is essentially unchanged (0.338 → 0.339) but Both BPs at val-best is lower (26.0% → 15.7%) because the higher-confidence threshold required to maximize F1 also drops more hits. The trade-off is favorable for the project goal (test performance), confirming the auto-derivation was the over-firing lever — POS_WEIGHT should track the *operating point*, not just inverse class ratio.

**Side observations:**
- Val/test gap is still large (val F1 0.339 vs test F1 0.149). The gap is *real* — not a threshold-tuning artifact, since both are at val-tuned threshold 0.9. Possible causes: (a) test set has structurally different SANTA configuration, (b) training data is too small (max_files=750 ≠ full dataset), (c) MAX_SEQ_LEN=10000 padding (~30%) creates BN boundary artifacts that don't transfer.
- Val ≥4kb Both BPs: 13.5% (down from 88.3% at thr=0.7 in #7, 16.8% in #6). The thr=0.7 number in #7 was peak-density artifact.

**Next:** Priority queue #2 — σ tuning. Try `LABEL_SIGMA = 20 → 10` to sharpen target peaks. Sharper targets should encourage more localized predictions (potentially fewer FPs per real BP) and let the F1 head room come from precision rather than recall. mean(y) will roughly halve (Gaussian integral 50 → 25), so cell-12's diagnostic will report a higher implied POS_WEIGHT, but POS_WEIGHT stays hardcoded at 70 from this run forward.

---

## 2026-05-04 #7 — `MAX_SEQ_LEN` 4000 → 10000 with re-derived `POS_WEIGHT`

**Hypothesis:** Run #6 surfaced strong evidence that 4000-bp truncation was clipping breakpoints on longer events: val 3-4kb gets 57.3% Both BPs but ≥4kb only 16.8%, and test (skewed long, evidenced by mean(y_test)=0.00163) was at 1.2% Both BPs. HIV-1 is ~9.7 kb. Bumping `MAX_SEQ_LEN` to 10000 should keep all breakpoints in-frame and mostly close the length-stratified gap.

**Change:** cell-3: `MAX_SEQ_LEN` 4000 → 10000, with a comment block explaining the truncation evidence. cell-12: added an automatic re-derivation of `POS_WEIGHT` from `mean(y * mask) / mean(mask)` after loading, since the positive rate shifts with the new sequence length. Empirically the re-derivation moved `POS_WEIGHT` 70 → 82.62 (mean_y_unmasked = 0.01196). cell-27: extended the threshold sweep to {0.2..0.99} since predictions now span the full sigmoid range and the val-best threshold can be much higher than the historical 0.3.

**Config snapshot:**
- max_files: 750  (2531 train / 415 val samples)
- LABEL_SIGMA: 20
- POS_WEIGHT: 82.62 (re-derived from data)
- LR: 1e-4
- Loss: weighted_bce
- Architecture: residual dilated stack (unchanged from #6)
- MAX_SEQ_LEN: 10000

**Results (at val-best threshold = 0.95 from extended sweep):**
- Best epoch: 23 (training continued 38 epochs total before early stop)
- Train PR-AUC at best epoch: ~0.220 (similar to #6)
- Val PR-AUC at best epoch: 0.192 (vs 0.208 in #6 — slight drop)
- Val F1 (peak-based): 0.338 (vs 0.238 in #6 — **+0.100 on val**)
- Val "Both BPs found": 26.0% (vs 24.8% in #6 — +1.2pp)
- Test F1 (peak-based): 0.113 (vs 0.122 in #6 — −0.009, within noise)
- Test "Both BPs found": 2.4%   ("One BP": 32.9%   "Missed": 64.6%)  (vs 1.2% in #6 — +1.2pp)
- Test precision / recall: 0.085 / 0.189
- Mean predicted peaks / event: 4.16 (test); 3.77 (val)
- Prediction range over val: [0.0000, 1.0000]

**Threshold sweep (val, n=415):**
- thr=0.20 P=0.080 R=0.953 F1=0.147  Both=93.3%  peaks=27.0
- thr=0.70 P=0.097 R=0.933 F1=0.173  Both=89.6%  peaks=21.4   (was the apparent sweet spot under the old 0.2-0.7 sweep)
- thr=0.95 P=0.275 R=0.496 F1=0.338  Both=26.0%  peaks=3.8    **(true val-best)**
- thr=0.99 P=0.309 R=0.324 F1=0.305  Both=7.5%   peaks=2.0

**Threshold sweep (test, n=82):**
- thr=0.20 P=0.020 R=0.384 F1=0.038  Both=25.6%  peaks=38.4
- thr=0.70 P=0.026 R=0.348 F1=0.048  Both=22.0%  peaks=28.6   (under-tuned, what initial run reported)
- thr=0.95 P=0.085 R=0.189 F1=0.113  Both=2.4%   peaks=4.2    (val-tuned)
- thr=0.99 P=0.175 R=0.165 F1=0.157  Both=1.2%   peaks=2.1    (highest test F1 in sweep, but not val-best)

**Verdict:** KEPT (under §6 qualitative criterion: val F1 +0.100, prediction range now fully [0.0000, 1.0000], threshold sweep reveals graded confidence)

**Why:** At val-tuned threshold, the test headline numbers are essentially flat (test F1 −0.009, test Both BPs +1.2pp — neither clears the §6 KEPT bars on its own). On val, the architecture now produces graded confidence with a clear F1 maximum at threshold=0.95, where test F1 is 0.113 and "Both BPs found" is 26.0%. The truncation hypothesis was directionally confirmed: at threshold=0.7 (which would have been picked under the old 0.2-0.7 sweep), val ≥4kb Both BPs jumped from 16.8% in #6 to 88.3% — but at that threshold the model fires 21+ peaks per genome and most are false positives. The peak-density inflation is the warning sign: the apparent +64.8pp val Both BPs gain at thr=0.7 was largely density artifact, not true localization. At a properly-calibrated threshold the gain shrinks to +1.2pp. The qualitative win is real (val F1 +0.100, calibration improved), but the truncation-fix-as-deliverable-Both-BPs win is much smaller than first reading suggested.

**Side observations:**
- POS_WEIGHT auto-derived 70 → 82.62. With residuals already finding signal, this may be over-aggressive — the model fires 27 peaks at thr=0.2 vs ~12 in #6, and the val/test gap widened (val F1 0.338 vs test F1 0.113 at val-tuned threshold).
- Test predictions concentrate higher than val (test min = 0.0875 vs val min = 0.0000) — the model is more confident overall on test, which means the val-tuned threshold may not transfer cleanly. Visible in the test sweep: test F1 keeps climbing past val-best 0.95 to 0.157 at threshold 0.99.
- Valid (non-padded) fraction dropped 0.992 → 0.691 — most genomes are <10000 bp. BatchNorm now sees ~30% padding zeros as part of its statistics; the model may be picking up boundary artifacts at the actual-sequence end.

**Next:** Run #8: revert `POS_WEIGHT` to 70 (disable auto-derivation, hardcode in cell-12) while keeping `MAX_SEQ_LEN=10000`. Single change, conservative test of whether the auto-derived 82.62 is the over-firing lever. If precision recovers without losing the val F1 gain, the auto-derivation is the issue and we standardize on 70. If it doesn't, the next move is masked BatchNorm or σ tuning.

🚩 **ESCALATION NOTE (per §8):** test "Both BPs found" improved less than the +20pp threshold once the threshold sweep is corrected (+1.2pp at val-tuned threshold), so the formal escalation does *not* fire. The pre-correction reading at thr=0.7 (test +20.8pp) was a sweep-bound artifact — flagging here so future readers don't recompute it from the raw thr=0.7 numbers and re-trigger the alarm.

---

## 2026-05-03 #5 — Dilated integration block (k=7, d=1..32) on BCE+bias-init baseline

**Hypothesis:** Replacing the 2-conv integration block (RF ~70 bp) with a 6-layer dilated stack `Conv1D(64, 7, dilation_rate=d)` for `d ∈ (1,2,4,8,16,32)` (RF ~410 bp) would let the model see flank composition on either side of a breakpoint and improve "Both BPs found" past the persistent 0% ceiling.

**Change:** cell-18: replaced two integration convs with the dilated stack. All other settings retained from #4 (bias init, weighted_bce(70), max_files=750).

**Config snapshot:**
- max_files: 750  (2531 train / 415 val samples)
- LABEL_SIGMA: 20
- POS_WEIGHT: 70
- LR: 1e-4
- Loss: weighted_bce
- Architecture: multi-scale (k=3,7,15,31) + dilated stack (k=7, d=1..32) + bias init

**Results:**
- Best epoch: 1 (training continued 16 epochs total before early stop; val_aupr drifted 0.187 → 0.179 over the run, with one ReduceLR halving)
- Train PR-AUC at best epoch: ~0.162
- Val PR-AUC at best epoch: 0.187
- Test F1 (peak-based): 0.195 (threshold=0.3, +/-200 bp)
- Test "Both BPs found": 0.0%   ("One BP": 29.3%   "Missed": 70.7%)
- Test precision / recall: 0.293 / 0.146
- Mean predicted peaks / event: 0.99 (test); 1.20 (val)
- Prediction range over val: [0.0098, 0.5825]

**Verdict:** INCONCLUSIVE

**Why:** The receptive-field-alone hypothesis was falsified — train PR-AUC stayed flat at ~0.162 (vs ~0.157 in #4), val PR-AUC nudged up by +0.012 (within noise on n=415), test F1 exactly unchanged at 0.195, and "Both BPs found" still 0%. But the failure mode is *specific*, not generic: best_epoch=1 with monotonic degradation under LR decay, and the prediction range *compressed* from [0.005, 0.92] in #4 to [0.01, 0.58] here. That signature points at optimization instability through 6 stacked BN+dilated blocks, not insufficient capacity. Receptive field is now adequate (~410 bp covers the +/-200 bp tolerance); gradient flow is the suspected limiter.

**Next:** Add WaveNet-style residual connections to the dilated stack (cell-18). Single change: each dilated block becomes `out = relu(BN(Conv1D(64,7,d=d)(x))) + x`, with a 1×1 projection on the first block to bridge the 256→64 channel mismatch from the multi-scale concat. Other settings unchanged. Expectation: best_epoch moves past 1, train PR-AUC climbs above 0.16, prediction range re-expands.

---

## 2026-05-03 #4 — BCE switch on original arch + bias init  *(historical)*

**Hypothesis:** Focal loss with continuous Gaussian targets has a `(1 - p_t)^γ` weighting that doesn't carry cleanly from binary y_true; plain weighted BCE should align loss with the metric.

**Change:** cell-15: added `weighted_bce(pos_weight)`. cell-18: switched compile to `loss=weighted_bce(POS_WEIGHT)`. cell-3: added `POS_WEIGHT=70.0`.

**Config snapshot:**
- max_files: 750  (2531 train / 415 val samples)
- LABEL_SIGMA: 20
- POS_WEIGHT: 70 (≈ inverse of mean(y)=0.0138)
- LR: 1e-4
- Loss: weighted_bce
- Architecture: original (multi-scale + Conv1D(128,7) + Conv1D(64,5) integration) + bias init

**Results:**
- Best epoch: 1 (val_aupr never improved past epoch 1; restored from there)
- Train PR-AUC at best epoch: 0.157
- Val PR-AUC at best epoch: 0.175
- Test F1 (peak-based): 0.195
- Test "Both BPs found": 0.0%   ("One BP": 29.3%   "Missed": 70.7%)
- Test precision / recall: 0.293 / 0.146
- Mean predicted peaks / event: 1.01
- Prediction range over val: [0.0047, 0.9199]

**Verdict:** KEPT (loss change retained for next experiments)

**Why:** Test F1 improved 0.167 → 0.195, prediction range expanded dramatically (peaks now reach 0.92), threshold sweep is more stable across 0.2–0.4. But train PR-AUC stayed flat at 0.158 across all 16 epochs while train loss dropped 2.5× — the architecture has saturated its ranking ability. "Both BPs found" stuck at 0% confirms the receptive-field hypothesis.

**Next:** Dilated integration block (RF ~70 → ~410 bp) as a clean single change against this baseline.

---

## 2026-05-03 #3 — Bias init on original arch with full dataset  *(historical)*

**Hypothesis:** RetinaNet-style negative bias init on the final sigmoid (`bias = -log((1-π)/π)`) escapes the sigmoid(0)=0.5 attractor that previous runs got stuck in.

**Change:** cell-18: `Conv1D(1, 1, sigmoid, bias_initializer=Constant(-4.595))` on the final layer. Original arch otherwise unchanged. (`max_files` was also bumped 100 → 750 in this run; this was a bundled change that should not have been combined — flagged for the record.)

**Config snapshot:**
- max_files: 750  (2531 train / 415 val samples)
- LABEL_SIGMA: 20
- Loss: focal_loss(α=0.25, γ=2.0)
- LR: 1e-4
- Architecture: original + bias init

**Results:**
- Best epoch: 20 (training continued for 35 epochs total before early stop)
- Train PR-AUC at best epoch: ~0.155
- Val PR-AUC at best epoch: 0.177
- Test F1 (peak-based): 0.167
- Test "Both BPs found": 0.0%   ("One BP": 25.6%   "Missed": 74.4%)
- Test precision / recall: 0.244 / 0.128
- Mean predicted peaks / event: 0.95
- Prediction range over val: [0.0217, 0.3505]

**Verdict:** KEPT (bias init retained going forward)

**Why:** Bias init unblocked training. Predictions broke out of the [0.41, 0.51] sigmoid-stuck range; val_aupr climbed monotonically across 20 epochs (rather than peaking at epoch 1 from random init); test F1 jumped 0.052 → 0.167. But "Both BPs found" stayed at 0% — model still emits ~1 peak per genome. Suggests architecture / RF is the next limiter.

**Next:** Switch focal → weighted BCE for cleaner pairing with Gaussian targets.

---

## 2026-05-03 #2 — Reverted dilated, kept mask + val_aupr  *(historical)*

**Hypothesis:** Reverting the dilated arch to the original integration block (while keeping mask + val_aupr from the previous run) would restore baseline performance — establishing a clean A/B point for the dilated change.

**Change:** cell-18: reverted to original (Conv1D(128,7) + Conv1D(64,5) integration). Kept padding mask in load_dataset/fit, kept val_aupr-monitoring callbacks.

**Config snapshot:**
- max_files: 100  (186 train / 34 val samples)
- LABEL_SIGMA: 20
- Loss: focal_loss
- LR: 1e-4
- Architecture: original (no bias init yet)

**Results:**
- Best epoch: 1 (random init); val_aupr 0.154 → 0.131 across 16 epochs
- Train PR-AUC at best epoch: 0.130
- Val PR-AUC at best epoch: 0.154
- Test F1 (peak-based): 0.052
- Test "Both BPs found": 3.7%   ("One BP": 37.8%   "Missed": 58.5%)
- Mean predicted peaks / event: 15.5 (nearly random — no real signal)
- Prediction range over val: [0.412, 0.512]

**Verdict:** REVERTED (this configuration as a baseline; problem identified, fixed in #3)

**Why:** Predictions stuck at sigmoid(0)≈0.5. Same failure mode as the dilated run — both architectures hit a stuck-at-init attractor under heavy class imbalance. Diagnosis: bias init on the final layer is required, not optional. Prior assumption that the original architecture didn't need it was wrong.

**Next:** Add bias init to the original arch (single change).

---

## 2026-05-03 #1 — Dilated arch + bias init bundle  *(historical, REVERTED)*

**Hypothesis:** Larger receptive field (~410 bp vs ~70) would let the model see flank composition on either side of a breakpoint and improve detection. Added bias init "while we're here" as a guardrail.

**Change:** cell-18: replaced two integration convs with `Conv1D(64, 7, dilation_rate=d)` for `d ∈ (1, 2, 4, 8, 16, 32)`, plus `bias_initializer=Constant(-4.595)` on the final layer.

**Config snapshot:**
- max_files: 100
- Architecture: dilated stack + bias init
- Loss: focal_loss

**Results:**
- Test F1: 0.052
- Test "Both BPs found": 3.7%
- Predictions stuck near sigmoid(0)=0.5; prediction range [0.477, 0.500]
- Train+val PR-AUC pinned at ~0.144 from epoch 1

**Verdict:** REVERTED — bundled change, untestable.

**Why:** Cannot attribute failure to dilated arch vs. some other issue because two changes were applied together. The advisor flagged this trap; future runs must be single-change.

**Next:** Revert architecture to original; if still failing, test bias init alone.

---

## Lessons distilled from runs #1–#5

- **Single-change-per-run is non-negotiable.** Run #1 lost a turn to bundling; runs #3 also bundled `max_files` and got away with it but shouldn't have.
- **Bias init is mandatory for the final sigmoid layer** under this level of class imbalance. Without it, every run gets stuck at sigmoid(0)=0.5 regardless of architecture or loss.
- **Train PR-AUC is the canary, not val PR-AUC.** When train PR-AUC is flat while train loss drops, the architecture has saturated. No amount of training-longer / regularising / loss-tuning will help — the issue is upstream of the loss.
- **"Both BPs found = 0%" has been the persistent ceiling** across all working runs. Every working run finds ~1 peak per event regardless of loss, threshold, or training duration. This is the real symptom and it points at receptive field — but receptive field expansion alone (#5) didn't fix it; gradient flow is the next suspect.
- **Prediction range is a useful canary for optimization health.** #4 had range [0.005, 0.92]; #5 with deeper stack collapsed to [0.01, 0.58]. When the range compresses after an architectural change, training is destabilizing, not improving. Keep an eye on this.
